In [ ]:
'''Name That LLM | Iteration 05: Name That LLM 05 Confidence Centering
Self-developed single-change experiment on the strong reference.
Neural attempts reported 0.557 and 0.556; their score-to-notebook
mapping is unknown, so this experiment explicitly uses notebook 03.
It does not claim that 03 is the higher-scoring run.

Hypothesis: average-linkage groups can contain contradictory pairs.
For each existing group, count within-group pairs whose distance
exceeds the existing clustering threshold. Centered evidence gets
weight 1 - 0.25 * disagreement_fraction, bounded to [0.75, 1].
Attenuate both frozen/LoRA centered heads and centered stylegram
replacement. Plain evidence, recovered groups, models, balance,
and Claude experts retain the reference settings. No new GPU pass.
This fixed hypothesis has not been validated on full-ensemble OOF.

GPU T4 x2, source Docker image, internet disabled.
Output: /kaggle/working/submission.parquet.
New experimental logic is implemented in classes; archived public
dependencies retain their original implementation and credits.
'''



In [ ]:
'''Name That LLM | Credits and Source Provenance
Vladislav Lavrentev: original LoRA, SmolLM3 training, TabPFN extension,
and the supplied inference notebook:
https://www.kaggle.com/code/lightningv08/0-557-lora-smollm3-tabpfn
Obstacle Developer: adaptive centered stylegram reference, pinned source:
https://www.kaggle.com/code/obstacledeveloper/local-candidate-adaptive-centered-stylegram?scriptVersionId=359313348
Lucas Boesen: backbone ensemble, features, attention heads, and centering:
https://www.kaggle.com/code/lucasboesen/lora-x-adaptive-centering
Zbigniew Kirchner: additional Qwen LoRA and Claude experts:
https://www.kaggle.com/code/zbigniewkirchner/0-552-adaptive-centering-own-lora-claude-gat
seyitkaangunes: public Claude API passages. pjmathematician: UltraBERT
fold weights. kirilltym: token-attention recipe. Upstream notices and
component licenses are preserved; this wrapper does not relicense them.
SmolLM3: Apache 2.0. TabPFN weights: Prior Labs Non-Commercial License.
Changes: OOP runtime, readable notebook sections, early asset checks,
final output validation, and quarantine of partial outputs on failure.
The original source is provided by the user; the current live notebook
has a newer method. We deliberately reproduce the supplied version.
'''



In [ ]:
'''Name That LLM | Imports
'''

import ast
from dataclasses import dataclass
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import time


In [ ]:
'''Name That LLM | ExperimentConfig
'''
@dataclass(frozen=True)
class ExperimentConfig:
    name: str
    full_ensemble: bool
    input_root: Path = Path('/kaggle/input')
    work_dir: Path = Path('/kaggle/working')
    temp_root: Path = Path('/kaggle/temp')
    timeout_seconds: int = 32400

    @property
    def source_dir(self):
        return self.temp_root / self.name


In [ ]:
'''Name That LLM | PublicSourceBundle
'''
class PublicSourceBundle:
    def __init__(self, digests):
        self.digests = digests
        self.sources = {}

    def add(self, name, source, crlf=False):
        if Path(name).name != name or name in self.sources:
            raise ValueError(f'Invalid or repeated source filename: {name}')
        if crlf is True:
            source = source.replace('\n', '\r\n')
        elif crlf:
            lines = source.splitlines(keepends=True)
            source = ''.join(line[:-1] + '\r\n' if i in crlf and line.endswith('\n') else line
                             for i, line in enumerate(lines))
        self.sources[name] = source

    def verify(self):
        if self.sources.keys() != self.digests.keys():
            raise ValueError('The source bundle is incomplete')
        for name, source in self.sources.items():
            if hashlib.sha256(source.encode()).hexdigest() != self.digests[name]:
                raise ValueError(f'Source integrity check failed: {name}')
            if name.endswith('.py'):
                ast.parse(source, filename=name)

    def materialize(self, directory):
        self.verify()
        directory.mkdir(parents=True, exist_ok=True)
        for name, source in self.sources.items():
            path = directory / name
            expected = source.encode()
            if path.exists() and path.read_bytes() != expected:
                raise FileExistsError(f'Refusing to overwrite different source: {path}')
            path.write_bytes(expected)


In [ ]:
'''Name That LLM | KagglePreflight
'''
class KagglePreflight:
    def __init__(self, config, bundle):
        self.config = config
        self.bundle = bundle

    def check(self):
        if not self.config.input_root.is_dir():
            raise RuntimeError('Run on Kaggle with the attached model inputs and two T4 GPUs')
        import pandas as pd
        import torch
        if torch.cuda.device_count() != 2 or any(
            'T4' not in torch.cuda.get_device_name(i) for i in range(2)
        ):
            raise RuntimeError('This reproduction requires GPU T4 x2')
        data_dirs = [
            self.config.input_root / 'competitions/name-that-llm',
            self.config.input_root / 'name-that-llm',
        ]
        data = next((p for p in data_dirs if (p / 'train.csv').is_file()), None)
        if data is None or not (data / 'test.csv').is_file():
            raise FileNotFoundError('Attach the Name That LLM competition data')
        train = pd.read_csv(data / 'train.csv')
        test = pd.read_csv(data / 'test.csv')
        if not {'id', 'text', 'model', 'prompt_id'}.issubset(train.columns):
            raise ValueError('Unexpected training data schema')
        if not {'id', 'text'}.issubset(test.columns) or test.empty:
            raise ValueError('Unexpected or empty test data')
        for frame in (train, test):
            if frame['id'].isna().any() or frame['id'].duplicated().any():
                raise ValueError('IDs must be unique and non-null')
        labels = json.loads(self.bundle.sources['smol-spec.json'])['classes']
        if set(train['model']) != set(labels):
            raise ValueError('Expected all 18 official author labels')
        patterns = [
            '**/glm4_9b_trunc25/config.json',
            '**/deberta-v3-large/config.json',
            '**/electra-large-discriminator/config.json',
            '**/desklib-detector/config.json',
            '**/lora24/final/COMPLETE.json',
            '**/lora_out/job_final/COMPLETE.json',
            '**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json',
            '**/tokattn_final.pt',
            '**/tokattn_debl23_final.pt',
            '**/ultrabert-380m-ntl/fold*/config.json',
            '**/ntl_extra_passages_claude.csv',
        ]
        missing = [pattern for pattern in patterns if not next(self.config.input_root.glob(pattern), None)]
        if missing:
            raise FileNotFoundError('Missing public ensemble assets: ' + ', '.join(missing))
        if self.config.full_ensemble:
            self._check_full_assets(data)
        return data, test, labels

    def _check_full_assets(self, data):
        spec = json.loads(self.bundle.sources['smol-spec.json'])
        actual = {name: importlib.metadata.version(name) for name in spec['core_versions']}
        if actual != spec['core_versions']:
            raise RuntimeError(f'Use the pinned source Docker image. Expected {spec["core_versions"]}; got {actual}')
        if hashlib.sha256((data / 'train.csv').read_bytes()).hexdigest() != spec['train_sha256']:
            raise ValueError('Official training data differs from the Smol checkpoint training data')
        optional = json.loads(self.bundle.sources['optional-config.json'])
        markers = {
            'asset_manifest.json': optional['compact_asset_manifest_sha256'],
            'smollm3_staging_complete.json': spec['base_manifest_sha256'],
        }
        for marker, digest in markers.items():
            matches = [p for p in self.config.input_root.rglob(marker)
                       if hashlib.sha256(p.read_bytes()).hexdigest() == digest]
            if len(matches) != 1:
                raise FileNotFoundError(f'Attach exactly one matching public Smol asset: {marker}')
        runtime = json.loads(self.bundle.sources['runtime_config.json'])
        tabular = list(self.config.input_root.rglob(runtime['tabpfn']['checkpoint_filename']))
        if len(tabular) != 1 or not (tabular[0].parent / 'runtime-addon-lock.txt').is_file():
            raise FileNotFoundError('Attach the pinned NTL Tabular Offline Assets dataset')


In [ ]:
'''Name That LLM | SubmissionArtifact
'''
class SubmissionArtifact:
    def __init__(self, config):
        self.path = config.work_dir / 'submission.parquet'

    def validate(self, test, labels):
        import pandas as pd
        if not self.path.is_file():
            raise FileNotFoundError(f'The ensemble did not produce {self.path}')
        result = pd.read_parquet(self.path)
        if list(result.columns) != ['id', 'model'] or len(result) != len(test):
            raise ValueError('Submission schema or row count differs from the official test data')
        if result.isna().any().any() or result['id'].duplicated().any():
            raise ValueError('Submission contains null values or repeated IDs')
        if not result['id'].equals(test['id'].reset_index(drop=True)):
            raise ValueError('Submission ID order differs from test.csv')
        if not result['model'].isin(labels).all():
            raise ValueError('Submission contains an unknown author label')
        return result

    def quarantine(self):
        if self.path.exists():
            destination = self.path.with_name(f'failed_run_{time.time_ns()}.parquet')
            self.path.rename(destination)


In [ ]:
'''Name That LLM | ReferenceEnsembleExperiment
'''
class ReferenceEnsembleExperiment:
    def __init__(self, config, bundle):
        self.config = config
        self.bundle = bundle
        self.artifact = SubmissionArtifact(config)

    def command(self):
        return [sys.executable, '-u', str(self.config.source_dir / 'ntl_reference.py')]

    def run(self):
        self.bundle.materialize(self.config.source_dir)
        data, test, labels = KagglePreflight(self.config, self.bundle).check()
        self.config.work_dir.mkdir(parents=True, exist_ok=True)
        if self.artifact.path.exists():
            raise FileExistsError('submission.parquet already exists; start a clean notebook session')
        environment = os.environ.copy()
        environment.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')
        started = time.monotonic()
        try:
            subprocess.run(self.command(), cwd=self.config.work_dir, env=environment,
                           check=True, timeout=self.config.timeout_seconds)
            submission = self.artifact.validate(test, labels)
        except BaseException:
            self.artifact.quarantine()
            raise
        summary = {
            'experiment': self.config.name,
            'method': 'full_smol_protected_tabpfn' if self.config.full_ensemble else 'reference_centered_stylegram',
            'test_rows': len(test),
            'elapsed_seconds': time.monotonic() - started,
            'reference_sha256': self.bundle.digests['ntl_reference.py'],
            'submission_sha256': hashlib.sha256(self.artifact.path.read_bytes()).hexdigest(),
            'source_modules_preserved': True,
            'fallback_used': False,
            'score_verified': False,
        }
        (self.config.work_dir / 'reproduction_summary.json').write_text(json.dumps(summary, indent=2))
        print(json.dumps(summary, indent=2))
        return submission


In [ ]:
'''Name That LLM | FullEnsembleExperiment
'''
class FullEnsembleExperiment(ReferenceEnsembleExperiment):
    def command(self):
        return [sys.executable, '-u', str(self.config.source_dir / 'run_candidate_optional.py')]


In [ ]:
'''Name That LLM | Frozen Source Manifest
'''
SOURCE_DIGESTS = {'ntl_reference.py': 'b856e736d5cb47d8fa309ce2acd5ce0588a4f91478bdeae99c491f31b7c26a4c', 'capture_runtime.py': '172aa2b5c336d6b277cdcf19d4d54e63749dc2b076f6b17a4cc56b71386275d6', 'claude_gate.py': '3fd1971a10eb15df21dd2c25acf6516d22e7fb07708b4cc0d4a872681fb37ae4', 'masked_features.py': 'ce6313975b0f8eb8a3b7c8ce9cd0d1656d0e16bd1f7340d50dbc42ec1ac991b2', 'tabular_worker.py': '71fa6fda5562f012a2d4b96bb8dcd862378f26fc39ece6068b97fce5e43af067', 'run_baseline.py': '28e4e8f93a14615532e07a0a141b06dd6630437c4cc1ed330b515436b2b8826a', 'protected_operator.py': '9d9f8a89b92935075131d1e3a26a7948cbbe6eb1495d5ca808329a5ecda3184a', 'QWEN35_LICENSE': 'bbedc3fda3305820b977265f01b8619d87570a6739de3a5582c3464840f1e57a', 'QWEN35_NOTICE.md': '5ce58b5e8ad74b3d9bf751e66b84721a0663311bd001ecc0d7a57d0c4c348c61', 'component.py': 'ab1df94cea1358d95125ed99f4fbe9606f7e0690ca6b6938ead419f884343f46', 'owned_path.py': '20dabafd3f423196863ce3d1fbd4caa8b196bd517a88422561615db003d1e620', 'run_component.py': '8d6ee8631ffa99baaf3f9fdb49e3d8a4dbd6dbe2284e2ea1617ea5965902052b', 'center_readout.py': '499b99f6ca94b6b4b8eb1fca15a4ef0519079194406441e70f8d0e2443068072', 'source_balance.py': 'da8682bc8298aed63c89607ceb08498a47a20ba7ab74752dddbdda996eab2351', 'compute_center.py': '3dbf7ca0c3769fbd4ade05346e4daeb791d589d7a7361e967f08d44c46999bbd', 'finish_center_protected.py': 'aa4e54e2fdc8ea4f209ae42138d6b808ac07c45bf232c68db0318c5df8da0a47', 'admission_preflight.py': '59de3c41b2a1146a11797008b80e77859a0282b0d8b14c6a152fafc9dba19eeb', 'run_parallel.py': '3ff30d27d811decbe0a19628a9933ae9a797f5191d303f7cd3257883ba66599f', 'run_candidate_optional.py': 'ab170cfa87abbfebcffb877a4b9db6b4cfa25944531b320e2e35da20822e013b', 'actual-diagnostic-audit.json': '8e76708fb1cca9239ffabfe7e8cd029b954d3957577f034156c3cfda4a9ba525', 'selection-evidence.json': '33c8a8a9e822ff608924e37007cfe93f40f9fd80f9386927097ca86cd95dbe67', 'runtime_config.json': 'b960e2660242c8b4419fe8f71bf1f3ee3ab3956384af61a84f716ed10b475290', 'optional-config.json': '06573fd83a350bf60cbf302bab73dc4c89c57fc11621d558d1518e4e8342abbe', 'smol-spec.json': 'a37480c78e4c53f1fed653482370a9c1867b34e775eac01a3d837a85d4fe9529'}
bundle = PublicSourceBundle(SOURCE_DIGESTS)


In [ ]:
'''Name That LLM | Upstream Module: ntl_reference.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('ntl_reference.py', r'''
# Name That LLM - v5: log-prob blend of
#   style-aware n-grams + LinearSVC | stylometric features + LightGBM |
#   Qwen3-8B-Base L7+L22 and Qwen2.5-7B L6+L20 (mean-pooled hidden states, layers concatenated) + LR |
#   public UltraBERT-380M 5-fold ensemble.
# CV 0.4927 (half-split held-out blend 0.4928) vs 0.450 for v4 (LB 0.436): paired +0.033 [+0.018, +0.048].
# Notebook settings: GPU T4 x2, internet off. Inputs: competition data; models qwen-lm/qwen-3 (transformers 8b-base),
# qwen-lm/qwen2.5 (transformers 7b); dataset pjmathematician/ultrabert-380m-ntl (fold weights, code = dmis-lab's).
# v7 = v6 + Qwen3-4B style-token-pooled head (L4+L7) + GLM-4-9B-0414 head (L8+L3, weights from a training-kernel output).
# v6 = v5 + two transductive steps (no labels used): test-set standardization of the LM-head features and
# Sinkhorn marginal balancing of the final posterior (alpha 0.5).
# v8 = v7 + capped adaptive prompt-centering for the 4 LM heads: test passages whose topic cluster (content-word
# TF-IDF, average linkage, cosine-distance cut ADAPT_THRESH) has ADAPT_MIN..ADAPT_MAX members use a prompt-centered
# twin of each head on cluster-centered features; everything else keeps the plain head. Never centers singletons
# (v1's failure) and never centers clusters > 18 (certainly mixed prompts). Simulated on held-out prompts (pooled
# 150-prompt test, v7 stack): +0.000 (1-2 passages/prompt) / +0.019 (1-5) / +0.034 (3-8) / +0.056 (all 18).
# v9 = v8 with the centered twins ADDED as extra blend evidence (plain heads kept for every passage) from cluster
# size 2 (as in the public 0.490 notebook). Pooled 150-prompt sim on the v7 stack, add>=2 vs plain:
# +0.024 (1-2 passages/prompt) / +0.042 (1-5) / +0.043 (3-8) / +0.052 (all 18); v8's replace>=4: 0 / +0.019 / +0.034 / +0.056.
# v10 = v9 with better test grouping and stronger centered evidence (pooled sim by session 4e): clusters from the
# average of the content-word TF-IDF cosine distance and the GLM-4-9B layer-24 cosine distance (mean-centered,
# L2-normalized) cut at 0.62 (at 0.65: same-prompt pair recall 0.89 vs 0.78 for TF-IDF alone, precision 0.97); centered
# twins with C x1.0 (was x0.4) and their evidence x1.5. Pooled-sim grid vs v9: 1-5 passages/prompt about +0.008,
# 1-2 about +0.003..+0.008, 3-8 / all-18 about +0.02; non-negative in every cell.
# v11 = v10 + a FROZEN DeBERTa-v3-large head (masked mean pool of layers 24+6, StandardScaler + LR C=0.005; CV 0.445
# alone, held-out +0.0061 on the v7 set, 18/20 splits; idea from Rudra's public fusion, which uses a frozen v3-base).
# Plain blend weights refit (L-BFGS, seed-0 OOF). The centered evidence is DECOUPLED from them: each twin gets an
# explicit weight TWIN_W (v10's 1.5 x v10 weight for the 4 LM twins, unchanged) plus a DeBERTa twin at 0.25.
# Pooled 150-prompt sim, 40 fresh draws, paired vs v10: 1-2 / 1-5 / 2-4 / 3-8 passages per prompt =
# +0.010 / +0.007 / +0.007 / +0.013 (SE 0.001-0.003; best of 6 variants, all of which beat v10 everywhere).
# v12 = v11 with better frozen encoders: DeBERTa-v3-large mean L23 + style-token pool L23 (0.450 alone) and
# ELECTRA-large discriminator mean L21 + style L24 (0.434). Held-out blend vs the v10 member set: +0.0157 (v11 set: +0.0061).
# v13 = v11 (same DeBERTa member and same twins) + frozen ELECTRA-large + frozen desklib AI-text detector (a DeBERTa-v3-large
# fine-tuned to detect AI text; mean L23 + style L23, 0.428 alone). Plain held-out +0.0088 vs the v11 set (20/20). In the
# pooled adapt sim v12 tied v11, while this set (v11 twins kept) beat v11 by +0.008/+0.000/+0.005/+0.003 (1-2/1-5/2-4/3-8).
# v13 public LB 0.532 (v8 0.499).
# v14 = v13 + a public LoRA classifier: Qwen3-8B-Base truncated to 24 layers, LoRA r16 on all linear layers, mean pool
# (first token masked) of layers 2/4/6/8/12/16/20/24, each with its own LayerNorm, concatenated -> linear head; trained on
# same-prompt triplets with plain + centered heads (kernel output lightningv08/ntl-lora24-fivefold-training, 'final' fit;
# its 5-fold OOF plain head = 0.4816 balanced accuracy). Here: inference only, plain head, as one more blend member.
# v15 = v14 + the LoRA's CENTERED head as a twin on clustered test rows (cluster-centered LoRA features -> centered
# readout, weight 0.15) + a learned token-attention head on frozen Qwen3.5-9B layer-18 token states (public recipe
# kirilltym/0-525-learned-token-attention, trained by kernel lucasboesen/name-that-llm-train-tokattn; OOF 0.394 alone).
# v16 = v15 + our own run of the LoRA24 recipe (seed 7, our folds; kernel lucasboesen/name-that-llm-train-lora24-s7; OOF
# 0.4804, argmax agreement with the public LoRA 0.69) + the same token-attention head on frozen DeBERTa-v3-large L23
# (kernel lucasboesen/name-that-llm-train-tokattn2; OOF 0.411). Held-out +0.0060 vs the v15 set (20/20 splits).
ADAPT, ADAPT_THRESH, ADAPT_MIN, ADAPT_MAX, ADAPT_C_MULT = True, 0.85, 2, 18, 1.0
TWIN_W = {'q3_8b_cat7+22': 1.5 * 0.219, 'q25_7b_cat6+20': 1.5 * 0.141, 'style4b': 1.5 * 0.083, 'glm9b': 1.5 * 0.110,
          'debv3l': 0.25}   # centered-evidence weight per twin (add mode); the first four = v10 exactly
CLUSTER_MODE, CLUSTER_THRESH_MIX, CLUSTER_LAYER = 'mixglm', 0.60, 24   # 'tfidf' = v9 clustering; 0.60: false-positive
#   pair rate at m=150 prompts sits between th0.58 (0.012, R 0.72) and th0.62 (0.059, R 0.85); tfidf 0.85: 0.053, R 0.80
ADAPT_MODE = 'add'     # 'add': plain + centered evidence on cluster rows; 'replace': centered only (v8)
SMOKE = False          # local code-path test: tiny models, 200 train rows, CPU ok
BALANCE_ALPHA = 0.5
TRANS_STD_MIN = 2      # min test rows for test-set standardization (2 so the 3-row interactive run exercises it)
W = {'stylegram': 1.516, 'q3_8b_cat7+22': 0.0, 'q25_7b_cat6+20': 0.0, 'style_lgb': 0.063, 'ultrabert': 0.095,
     'style4b': 0.0, 'glm9b': 0.0, 'debv3l': 0.05, 'electl': 0.032, 'desk': 0.132, 'lora': 0.177, 'tokattn': 0.153,
     'lora2': 0.113, 'lora3': 0.177, 'tokattn_deb': 0.185}
# L-BFGS on seed-0 OOF; held-out +0.0060 vs the v15 set (20/20 splits). Pooled adapt sim vs v15 (60 fresh draws): a TIE
# (+0.000 / +0.001 / +0.002 / -0.001), so v16 is submitted as an EXPERIMENT (does a plain-only gain transfer to the LB?).
LORA = dict(enabled=True, layers=[2, 4, 6, 8, 12, 16, 20, 24], max_len=447, bs=8)
LORAS = [  # (key, checkpoint marker glob, norms file, anchor-features file): all share the Qwen3-8B-Base 24-layer trunk
    ('lora', '**/lora24/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz'),   # public lightningv08 run
    ('lora2', '**/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz'),   # our seed-7 run
    ('lora3', '**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz'),   # zbigniewkirchner: + Claude head
]
TWIN_W_LORA = {'lora': 0.15, 'lora3': 0.15}   # centered-readout evidence on clustered rows (add mode)
TOKATTN = dict(enabled=True, bs=8)
TOKATTN_HEADS = [  # (key, head file from a training-kernel output, backbone)
    ('tokattn', 'tokattn_final.pt', 'qwen3_5'),            # Qwen3.5-9B L18 (decoder, fp16)
    ('tokattn_deb', 'tokattn_debl23_final.pt', 'deberta'),   # DeBERTa-v3-large L23 (encoder, fp32)
]
ENCODERS = [  # (key, Kaggle input dir name holding fold0..fold4, config model_type, trust_remote_code, local fallbacks)
    ('ultrabert', 'ultrabert-380m-ntl', 'ultrabert', True,
     ['external/ultrabert-380m-ntl', '../../external/ultrabert-380m-ntl']),
]
LLM_HEADS = [  # (key, model_type, hidden_size, layers (concatenated), C, smoke HF id, pooling)
    ('q3_8b_cat7+22', 'qwen3', 4096, [7, 22], 0.0025, 'Qwen/Qwen3-0.6B-Base', 'mean'),
    ('q25_7b_cat6+20', 'qwen2', 3584, [6, 20], 0.0025, 'Qwen/Qwen2.5-0.5B', 'mean'),
    ('style4b', 'qwen3', 2560, [4, 7], 0.0025, 'Qwen/Qwen3-0.6B-Base', 'style'),   # Qwen3-4B-Base, style-token pooling
    ('glm9b', 'glm4', 4096, [8, 3], 0.0025, 'Qwen/Qwen2.5-0.5B', 'mean'),         # GLM-4-9B-0414 truncated to 25 layers
]
FROZEN_ENC = [  # (key, input dir name, [(layer, pool)] concatenated, C, smoke HF id): bidirectional encoders, fp32
    ('debv3l', 'deberta-v3-large', [(24, 'mean'), (6, 'mean')], 0.005, 'microsoft/deberta-v3-xsmall'),   # = v11 member
    ('electl', 'electra-large-discriminator', [(21, 'mean'), (24, 'style')], 0.005, 'google/electra-small-discriminator'),
    ('desk', 'desklib-detector', [(23, 'mean'), (23, 'style')], 0.005, 'microsoft/deberta-v3-xsmall'),
]   # weights: kernel outputs name-that-llm-extract-encoders2, -electra-large-weights, -desklib-weights (found by dir name:
#     desklib and DeBERTa-v3-large share model_type/hidden size)

import gc
import glob
import json
import os
import pathlib
import re
import time

import numpy as np
import pandas as pd
import torch
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import hstack
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from transformers import AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer

T0 = time.time()
LABELS = sorted([
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
K = len(LABELS)
DATA_DIR = next(p for p in [pathlib.Path('/kaggle/input/competitions/name-that-llm'),
                            pathlib.Path('/kaggle/input/name-that-llm'),
                            pathlib.Path('data')] if (p / 'train.csv').exists())
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
if SMOKE:
    train = train.sample(200, random_state=0).reset_index(drop=True)
test['text'] = test['text'].fillna('').astype(str)
y = train['model'].map({l: i for i, l in enumerate(LABELS)}).values
TR, TE = train['text'].tolist(), test['text'].tolist()
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
NGPU = torch.cuda.device_count()
print('train', train.shape, 'test', test.shape, 'gpus', NGPU, flush=True)
LOGIT = np.zeros((len(test), K))


def add(key, prob):
    LOGIT[:] += W[key] * np.log(np.clip(prob, 1e-9, 1))
    print(f'{key} done ({time.time() - T0:.0f}s)', flush=True)


def topic_clusters(texts, thresh):
    """Average-linkage clusters on cosine distance of content-word TF-IDF (stop words removed)."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    X = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit_transform(texts)
    D = np.clip(1 - (X @ X.T).toarray(), 0, None)
    np.fill_diagonal(D, 0)
    return fcluster(linkage(squareform(D, checks=False), method='average'), thresh, criterion='distance')


def group_center(X, groups):
    X = np.asarray(X, dtype=np.float32).copy()
    for g in np.unique(groups):
        i = groups == g
        X[i] -= X[i].mean(0)
    return X


def mixed_clusters(texts, E, thresh):
    """Average-linkage clusters on the mean of the TF-IDF cosine distance and the cosine distance of the
    mean-centered, L2-normalized embeddings E (GLM-4-9B layer 24)."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    X = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit_transform(texts)
    Dt = np.clip(1 - (X @ X.T).toarray(), 0, None)
    Z = np.asarray(E, dtype=np.float64)
    Z = Z - Z.mean(0)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
    D = (Dt + np.clip(1 - Z @ Z.T, 0, None)) / 2
    np.fill_diagonal(D, 0)
    return fcluster(linkage(squareform(D, checks=False), method='average'), thresh, criterion='distance')


def compute_groups(E_te=None, E_selftest=None):
    """Adaptive-centering targets: test passages in clusters of size ADAPT_MIN..ADAPT_MAX (+ tiny-test self-test)."""
    global G_TE, ADAPT_ROWS, SELFTEST
    use_mix = CLUSTER_MODE == 'mixglm' and E_te is not None
    G_TE = (mixed_clusters(TE, E_te, CLUSTER_THRESH_MIX) if use_mix else topic_clusters(TE, ADAPT_THRESH)) \
        if ADAPT else np.ones(len(TE), dtype=int)
    sz = np.bincount(G_TE)[G_TE]
    ADAPT_ROWS = ADAPT & (sz >= ADAPT_MIN) & (sz <= ADAPT_MAX)
    print('clustering', 'mixglm' if use_mix else 'tfidf', '| test cluster sizes (passages): 1:', int((sz == 1).sum()),
          ' 2-3:', int(((sz >= 2) & (sz <= 3)).sum()), ' 4-18:', int(((sz >= 4) & (sz <= 18)).sum()),
          ' >18:', int((sz > 18).sum()), ' | adapt rows:', int(ADAPT_ROWS.sum()), ADAPT_MODE, flush=True)
    SELFTEST = None
    if ADAPT and len(TE) < 50:   # interactive 3-row test: exercise the adapt branch on train rows (output discarded)
        st = np.where(train['prompt_id'].isin(train['prompt_id'].drop_duplicates().iloc[:3]).values)[0]
        txt = [TR[i] for i in st]
        g = mixed_clusters(txt, E_selftest[st], CLUSTER_THRESH_MIX) if (use_mix and E_selftest is not None) \
            else topic_clusters(txt, ADAPT_THRESH)
        s = np.bincount(g)[g]
        keep = (s >= ADAPT_MIN) & (s <= ADAPT_MAX)
        SELFTEST = (st[keep], g[keep])
        print('adapt self-test rows:', int(keep.sum()), 'of', len(st), flush=True)
        if use_mix and E_selftest is not None:   # clustering QA on a train-built pseudo-test (1-5 passages/prompt)
            rng = np.random.RandomState(0)
            pid = train['prompt_id'].values
            qa = np.concatenate([rng.choice(np.where(pid == p)[0], min(rng.randint(1, 6), int((pid == p).sum())),
                                            replace=False) for p in np.unique(pid)])
            gq = mixed_clusters([TR[i] for i in qa], E_selftest[qa], CLUSTER_THRESH_MIX)
            sq = np.bincount(gq)[gq]
            same_c = gq[:, None] == gq[None, :]
            same_p = pid[qa][:, None] == pid[qa][None, :]
            off = ~np.eye(len(qa), dtype=bool)
            tp = (same_c & same_p & off).sum()
            print(f'clustering QA on {len(qa)} train passages (1-5/prompt): pair precision '
                  f'{tp / max((same_c & off).sum(), 1):.3f} recall {tp / max((same_p & off).sum(), 1):.3f} | sizes 1:',
                  int((sq == 1).sum()), ' 2-3:', int(((sq >= 2) & (sq <= 3)).sum()), ' 4-18:',
                  int(((sq >= 4) & (sq <= 18)).sum()), ' >18:', int((sq > 18).sum()), flush=True)


G_TE, ADAPT_ROWS, SELFTEST = None, None, None   # set by compute_groups() after the GLM pass (section 3)


# ------------------------------------------------ 1) style-aware n-grams + LinearSVC
# ADAPTIVE_STYLEGRAM_CENTERING_CANDIDATE_V1
# Keep the original stylegram head as fallback; replace it only on text-only
# clusters inferred from this test pool. CV used prompt-held-out folds.
FUNCTION_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
SPEC = [('char', 3, 5, False, 250000, 1.0), ('word', 1, 3, False, 180000, 0.7),
        ('shape', 2, 6, False, 60000, 0.5), ('function', 1, 3, False, 50000, 0.5)]


def shape_text(text):
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", text)


def function_text(text):
    return " ".join(t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else "CONTENT"
                    for t in re.findall(r"\w+|[^\w\s]", text))


STYLEGRAM_CLUSTER_THRESHOLD = 0.90
STYLEGRAM_CLUSTER_MIN = 2
STYLEGRAM_CLUSTER_MAX = 18


def _stylegram_center_sparse(X, group_ids, min_size=2, max_size=18):
    """Subtract sparse group means; leave groups outside the size range unchanged."""
    from scipy.sparse import csr_matrix, diags
    X = csr_matrix(X)
    group_ids = np.asarray(group_ids)
    _, inverse, counts = np.unique(group_ids, return_inverse=True, return_counts=True)
    valid = np.flatnonzero((counts >= min_size) & (counts <= max_size))
    if not len(valid):
        return X.copy()
    remap = np.full(len(counts), -1, dtype=np.int32)
    remap[valid] = np.arange(len(valid), dtype=np.int32)
    mapped = remap[inverse]
    rows = np.flatnonzero(mapped >= 0)
    membership = csr_matrix(
        (np.ones(len(rows), dtype=X.dtype), (rows, mapped[rows])),
        shape=(X.shape[0], len(valid)), dtype=X.dtype)
    means = diags((1.0 / counts[valid]).astype(X.dtype)) @ (membership.T @ X)
    return (X - membership @ means).tocsr()


def _stylegram_cluster_ids(texts, threshold=STYLEGRAM_CLUSTER_THRESHOLD):
    """Median-normalized content-word TF-IDF distances, matching local CV."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    _tf = TfidfVectorizer(stop_words='english', sublinear_tf=True, min_df=1,
                          max_features=100_000, dtype=np.float32)
    _xt = _tf.fit_transform(texts)
    _dt = np.clip(1.0 - (_xt @ _xt.T).toarray(), 0.0, 1.0)
    np.fill_diagonal(_dt, 0.0)
    _upper = np.triu_indices(len(texts), 1)
    _median = max(float(np.median(_dt[_upper])), 1e-6)
    _dt /= _median
    np.fill_diagonal(_dt, 0.0)
    return fcluster(linkage(squareform(_dt, checks=False), method='average'),
                    float(threshold), criterion='distance')


A, B = [], []
for analyzer, lo, hi, lower, maxf, w in SPEC:
    tf = shape_text if analyzer == 'shape' else function_text if analyzer == 'function' else str
    actual = 'char' if analyzer == 'shape' else 'word' if analyzer == 'function' else analyzer
    opts = dict(analyzer=actual, ngram_range=(lo, hi), lowercase=lower, sublinear_tf=True, min_df=2,
                max_features=maxf, dtype=np.float64)
    if actual == 'word':
        opts['token_pattern'] = r"(?u)\b\w+\b|[^\w\s]"
    v = TfidfVectorizer(**opts)
    A.append(v.fit_transform([tf(t) for t in TR]) * w)
    B.append(v.transform([tf(t) for t in TE]) * w)
X_stylegram_train = hstack(A, format='csr')
X_stylegram_test = hstack(B, format='csr')

# Original plain model and probability mapping.
svc = LinearSVC(C=0.5, random_state=42, max_iter=4000).fit(X_stylegram_train, y)
s = svc.decision_function(X_stylegram_test)
s = np.exp(s - s.max(1, keepdims=True))
p = np.zeros((len(TE), K))
p[:, svc.classes_] = s / s.sum(1, keepdims=True)

# Centered twin is trained on known prompt groups, then used only for inferred
# test groups. The test pool's prompt IDs/labels are never read.
if ADAPT:
    X_stylegram_train_centered = _stylegram_center_sparse(
        X_stylegram_train, train['prompt_id'].values,
        STYLEGRAM_CLUSTER_MIN, STYLEGRAM_CLUSTER_MAX)
    svc_center = LinearSVC(C=0.5, random_state=42, max_iter=4000).fit(
        X_stylegram_train_centered, y)
    G_STYLEGRAM = _stylegram_cluster_ids(TE, STYLEGRAM_CLUSTER_THRESHOLD)
    _stylegram_sizes = np.bincount(G_STYLEGRAM)[G_STYLEGRAM]
    _stylegram_rows = ((_stylegram_sizes >= STYLEGRAM_CLUSTER_MIN) &
                       (_stylegram_sizes <= STYLEGRAM_CLUSTER_MAX))
    if _stylegram_rows.any():
        X_stylegram_test_centered = _stylegram_center_sparse(
            X_stylegram_test[_stylegram_rows], G_STYLEGRAM[_stylegram_rows],
            STYLEGRAM_CLUSTER_MIN, STYLEGRAM_CLUSTER_MAX)
        _s_center = svc_center.decision_function(X_stylegram_test_centered)
        _s_center = np.exp(_s_center - _s_center.max(1, keepdims=True))
        _p_center = np.zeros((int(_stylegram_rows.sum()), K))
        _p_center[:, svc_center.classes_] = _s_center / _s_center.sum(1, keepdims=True)
        p[_stylegram_rows] = _p_center
        del X_stylegram_test_centered, _s_center, _p_center
    print(f'stylegram centered replacement: {_stylegram_rows.sum()}/{len(TE)} rows; '
          f'eligible clusters {STYLEGRAM_CLUSTER_MIN}-{STYLEGRAM_CLUSTER_MAX}, '
          f'median-normalized TF-IDF threshold {STYLEGRAM_CLUSTER_THRESHOLD:.2f}', flush=True)
    del X_stylegram_train_centered, svc_center

add('stylegram', p)
del A, B, svc, X_stylegram_train, X_stylegram_test, s, p

# ------------------------------------------------ 1b) within-Claude SVC on original Claude rows + free public Claude API passages
# (dataset seyitkaangunes/ntl-extra-api-passages). Used only to choose WHICH Claude after the final blend says "Claude".
# Local GroupKFold CV on the stylegram model: 0.3845 -> 0.3967 (Claude recall 0.393 -> 0.438, other classes unchanged).
CLAUDE_GATE_LAM = 1.5
CLAUDE_LORA_LAM = 1.0
LC_TE = None
_api_hits = glob.glob('/kaggle/input/**/ntl_extra_passages_claude.csv', recursive=True)
CLAUDE_IDX = np.array([i for i, l in enumerate(LABELS) if l.startswith('claude')])
DC_TE = None
if _api_hits:
    _api = pd.read_csv(_api_hits[0])
    _m = np.isin(y, CLAUDE_IDX)
    _ft = [t for t, k in zip(TR, _m) if k] + _api['text'].astype(str).tolist()
    _fy = np.concatenate([y[_m], _api['model'].map({l: i for i, l in enumerate(LABELS)}).values])
    A, B = [], []
    for analyzer, lo, hi, lower, maxf, w in SPEC:
        tf = shape_text if analyzer == 'shape' else function_text if analyzer == 'function' else str
        actual = 'char' if analyzer == 'shape' else 'word' if analyzer == 'function' else analyzer
        opts = dict(analyzer=actual, ngram_range=(lo, hi), lowercase=lower, sublinear_tf=True, min_df=2,
                    max_features=maxf, dtype=np.float64)
        if actual == 'word':
            opts['token_pattern'] = r"(?u)\b\w+\b|[^\w\s]"
        v = TfidfVectorizer(**opts)
        A.append(v.fit_transform([tf(t) for t in _ft]) * w)
        B.append(v.transform([tf(t) for t in TE]) * w)
    _svc = LinearSVC(C=0.5, random_state=42, max_iter=4000).fit(hstack(A, format='csr'), _fy)
    assert list(_svc.classes_) == list(CLAUDE_IDX)
    DC_TE = _svc.decision_function(hstack(B, format='csr'))
    print('claude gate model trained on', len(_ft), 'rows', flush=True)
    del A, B, _svc, _api
else:
    print('WARNING: Claude API passages not attached, gate disabled', flush=True)


# ------------------------------------------------ 2) stylometric features + LightGBM
PUNCT = list(',.;:!?()[]"\'-') + ['—', '–', '‘', '’', '“', '”', '…', '*', '/', '%']
FUNC = sorted(ENGLISH_STOP_WORDS)
SENT = re.compile(r'(?<=[.!?])\s+')
WORD = re.compile(r"[A-Za-z’']+")


def stylometric_features(t):
    words = WORD.findall(t)
    nw, nc = max(len(words), 1), max(len(t), 1)
    sents = [x for x in SENT.split(t.strip()) if x]
    sl = np.array([len(WORD.findall(x)) for x in sents]) if sents else np.array([0])
    wl = np.array([len(x) for x in words]) if words else np.array([0])
    lw = [x.lower() for x in words]
    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(),
         (wl >= 9).mean(), len(set(lw)) / nw, sum(c.isupper() for c in t) / nc, sum(c.isdigit() for c in t) / nc,
         sum(ord(c) > 127 for c in t) / nc, t.count(', ') / nw, t.count(' and ') / nw, t.count(' but ') / nw,
         sum(1 for x in sents if x[:1].isupper() and x.split()[0].lower() in ('this', 'these', 'it', 'they')) / max(len(sents), 1),
         sum(1 for x in words if x.endswith('ly')) / nw, sum(1 for x in words if x.endswith('ing')) / nw,
         sum(1 for x in words if x.endswith('tion')) / nw]
    f += [t.count(q) / nw for q in PUNCT]
    cnt = {}
    for x in lw:
        cnt[x] = cnt.get(x, 0) + 1
    f += [cnt.get(x, 0) / nw for x in FUNC]
    return f


import lightgbm as lgb
lg = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8,
                        subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0, random_state=0, verbose=-1, n_jobs=4)
lg.fit(np.array([stylometric_features(t) for t in TR], dtype=np.float32), y)
p = np.zeros((len(TE), K))
p[:, lg.classes_] = lg.predict_proba(np.array([stylometric_features(t) for t in TE], dtype=np.float32))
add('style_lgb', p)


# ------------------------------------------------ 3) frozen LLM heads (truncated to layer+1, fp16 on GPU)
def find_model_dir(model_type, hidden):
    for cfg in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if c.get('model_type') == model_type and c.get('hidden_size') == hidden:
            return os.path.dirname(cfg)
    return None


def load_truncated(path, keep, dtype):
    cfg = AutoConfig.from_pretrained(path)
    cfg.num_hidden_layers = keep
    if getattr(cfg, 'layer_types', None):
        cfg.layer_types = cfg.layer_types[:keep]
    kw = dict(config=cfg, dtype=dtype)
    if DEV == 'cuda':
        kw['device_map'] = 'auto'
    return AutoModel.from_pretrained(path, **kw).eval()


def style_mask(tok):
    """Vocab mask of 'style' tokens: decoded piece is an English stop word or pure punctuation."""
    func = frozenset(ENGLISH_STOP_WORDS)
    pieces = [tok.decode([i]) for i in range(len(tok))]
    return torch.tensor([(p.strip().lower() in func) or bool(re.fullmatch(r'\s*[^\w\s]+\s*', p)) for p in pieces])


@torch.no_grad()
def embed_layers(model, tok, texts, layers, bs=8, max_len=512, pool='mean', smask=None):
    """Pooled hidden_states[L] for each L in layers, concatenated -> (n, len(layers) * hidden).
    pool='mean': all tokens; pool='style': only tokens in smask (function words / punctuation)."""
    H = model.config.hidden_size
    out = np.zeros((len(texts), len(layers) * H), dtype=np.float32)
    order = np.argsort([len(t) for t in texts])
    dev = next(model.parameters()).device
    bad = 0
    for b0 in range(0, len(texts), bs):
        idx = order[b0:b0 + bs]
        batch = [(tok.bos_token + texts[i]) if tok.bos_token else texts[i] for i in idx]
        enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_len,
                  add_special_tokens=False).to(dev)
        hs = model(**enc, output_hidden_states=True).hidden_states
        if not all(torch.isfinite(hs[L]).all() for L in layers):   # fp16 overflow guard: switch to bf16, redo
            bad += 1
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True).hidden_states
        for li, L in enumerate(layers):
            h = hs[L]
            m = enc['attention_mask'].float()
            if pool == 'style':
                m = m * smask.to(m.device)[enc['input_ids']].float()
            m = m.unsqueeze(-1).to(h.device)
            out[idx, li * H:(li + 1) * H] = ((h.float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
        del hs, enc
    if bad:
        print('  non-finite fp16 batches redone in bf16:', bad, flush=True)
    return np.nan_to_num(out)


def extract(key, mt, hid, lays, smoke_id, pool):
    """Pooled hidden states of layers `lays` (concatenated) for train and test, model truncated to max(lays)+1."""
    path = smoke_id if SMOKE else find_model_dir(mt, hid)
    print(key, path, lays, pool, flush=True)
    assert path is not None, f'model for {key} ({mt}, hidden {hid}) not found under /kaggle/input'
    tok = AutoTokenizer.from_pretrained(path)
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    smask = style_mask(tok) if pool == 'style' else None
    dtype = torch.float16 if DEV == 'cuda' else torch.float32
    model = load_truncated(path, max(lays) + 1, dtype)
    if SMOKE:   # truncation must not change hidden_states[L]
        full = AutoModel.from_pretrained(path, dtype=dtype).eval()
        e1 = embed_layers(model, tok, TE, lays, pool=pool, smask=smask)
        e2 = embed_layers(full, tok, TE, lays, pool=pool, smask=smask)
        print('  truncation max abs diff', float(np.abs(e1 - e2).max()), flush=True)
        assert np.allclose(e1, e2, atol=1e-4)
        del full
    Etr = embed_layers(model, tok, TR, lays, pool=pool, smask=smask)
    Ete = embed_layers(model, tok, TE, lays, pool=pool, smask=smask)
    del model, tok
    gc.collect()
    if DEV == 'cuda':
        torch.cuda.empty_cache()
    return Etr, Ete


def smoke_layers(layers):
    return [min(L, 3 + i) for i, L in enumerate(layers)] if SMOKE else list(layers)


# GLM pre-pass: the GLM head's layers plus the clustering layer in one forward pass; then build the test groups.
CACHE = {}
_glm = next((h for h in LLM_HEADS if h[1] == 'glm4'), None)
if _glm is not None and CLUSTER_MODE == 'mixglm' and ADAPT:
    _gl = smoke_layers(_glm[3])
    _cl = 5 if SMOKE else CLUSTER_LAYER
    _Etr, _Ete = extract(_glm[0], _glm[1], _glm[2], _gl + [_cl], _glm[5], _glm[6])
    _H = _Etr.shape[1] // (len(_gl) + 1)
    CACHE[_glm[0]] = (_Etr[:, :len(_gl) * _H], _Ete[:, :len(_gl) * _H])
    compute_groups(_Ete[:, len(_gl) * _H:], _Etr[:, len(_gl) * _H:])
    del _Etr, _Ete
else:
    compute_groups()

def fit_head(key, Etr, Ete, C):
    """StandardScaler + LR head on pooled features; optional prompt-centered twin on clustered test rows."""
    sc = StandardScaler().fit(Etr)
    clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Etr), y)
    # transductive standardization: test features scaled with TEST mean/std (labels never used); CV: never
    # negative, +0.002..+0.01 depending on test size. Train stats fallback for tiny tests (3-row interactive file).
    Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-6) if len(Ete) >= TRANS_STD_MIN else sc.transform(Ete)
    p = np.zeros((len(TE), K))
    p[:, clf.classes_] = clf.predict_proba(Xte)
    if ADAPT and key in TWIN_W:   # prompt-centered twin, used only on test passages in clusters of ADAPT_MIN..ADAPT_MAX
        Xc = group_center(Etr, train['prompt_id'].values)
        csc = StandardScaler().fit(Xc)
        cclf = LogisticRegression(C=ADAPT_C_MULT * C, max_iter=5000).fit(csc.transform(Xc), y)
        if ADAPT_ROWS.any():
            pc = np.zeros((int(ADAPT_ROWS.sum()), K))
            pc[:, cclf.classes_] = cclf.predict_proba(csc.transform(group_center(Ete[ADAPT_ROWS], G_TE[ADAPT_ROWS])))
            if ADAPT_MODE == 'replace':
                p[ADAPT_ROWS] = pc
            else:   # add the centered head's evidence on top of the plain head (applied after add(key, p))
                LOGIT[ADAPT_ROWS] += TWIN_W[key] * np.log(np.clip(pc, 1e-9, 1))
        if SELFTEST is not None and len(SELFTEST[0]):
            ps = cclf.predict_proba(csc.transform(group_center(Etr[SELFTEST[0]], SELFTEST[1])))
            print(f'  adapt self-test ({len(SELFTEST[0])} train rows, seen in training): '
                  f'acc {np.mean(cclf.classes_[ps.argmax(1)] == y[SELFTEST[0]]):.2f}', flush=True)
    add(key, p)


for key, mt, hid, layers, C, smoke_id, pool in LLM_HEADS:
    if key in CACHE:
        Etr, Ete = CACHE.pop(key)
    else:
        Etr, Ete = extract(key, mt, hid, smoke_layers(layers), smoke_id, pool)
    fit_head(key, Etr, Ete, C)


# ------------------------------------------------ 3b) frozen bidirectional encoders (fp32, masked mean pool incl. [CLS]/[SEP])
@torch.no_grad()
def embed_encoder(model, tok, texts, parts, smask, bs=16, max_len=512):
    """parts = [(layer, 'mean'|'style')]: masked mean of hidden_states[layer] over all tokens (incl. [CLS]/[SEP]) or
    over style tokens only (stop words / punctuation), concatenated -> (n, len(parts) * hidden)."""
    H = model.config.hidden_size
    out = np.zeros((len(texts), len(parts) * H), dtype=np.float32)
    order = np.argsort([len(t) for t in texts])
    dev = next(model.parameters()).device
    smask = smask.to(dev)
    for b0 in range(0, len(texts), bs):
        idx = order[b0:b0 + bs]
        enc = tok([texts[i] for i in idx], padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(dev)
        hs = model(**enc, output_hidden_states=True).hidden_states
        m = enc['attention_mask'].unsqueeze(-1).float()
        ms = m * smask[enc['input_ids']].unsqueeze(-1).float()
        for li, (L, pool) in enumerate(parts):
            w = ms if pool == 'style' else m
            out[idx, li * H:(li + 1) * H] = ((hs[L].float() * w).sum(1) / w.sum(1).clamp(min=1)).cpu().numpy()
        del hs, enc
    return np.nan_to_num(out)


def find_dir(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}/config.json', recursive=True))
    return os.path.dirname(hits[0]) if hits else None


for key, dname, parts, C, smoke_id in FROZEN_ENC:
    path = smoke_id if SMOKE else find_dir(dname)
    print(key, path, parts, flush=True)
    assert path is not None, f'encoder for {key} (dir {dname}) not found under /kaggle/input'
    etok = AutoTokenizer.from_pretrained(path)
    emodel = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()
    if SMOKE:
        parts = [(min(L, 3 + i), pool) for i, (L, pool) in enumerate(parts)]
    sm = style_mask(etok)
    Etr, Ete = embed_encoder(emodel, etok, TR, parts, sm), embed_encoder(emodel, etok, TE, parts, sm)
    del emodel, etok
    gc.collect()
    if DEV == 'cuda':
        torch.cuda.empty_cache()
    fit_head(key, Etr, Ete, C)

# ------------------------------------------------ 3c) public LoRA classifier (inference only, plain head)
@torch.no_grad()
def run_loras(texts, parity_texts):
    """Pooled LoRA features of every checkpoint in LORAS (one Qwen3-8B-Base trunk, adapters swapped). For each: plain
    readout -> blend member W[key]; centered readout on cluster-centered features -> twin evidence TWIN_W_LORA[key]."""
    import importlib.util as _iu
    _fs = _iu.find_spec   # Kaggle ships torchao 0.10, which peft rejects; hide it (unused)
    _iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _fs(name, *a, **k)
    from peft import PeftModel
    import peft.import_utils as _pi
    _pi.is_torchao_available = lambda *a, **k: False
    from safetensors.torch import load_file
    base_dir = find_model_dir('qwen3', 4096)
    cfg = AutoConfig.from_pretrained(base_dir)
    cfg.num_hidden_layers, cfg.use_cache = 24, False
    if getattr(cfg, 'layer_types', None) is not None:
        cfg.layer_types = cfg.layer_types[:24]
    trunk = AutoModel.from_pretrained(base_dir, config=cfg, dtype=torch.float16, device_map='auto').eval()
    L = LORA['layers']
    for key, marker, norms_file, anchors_file in LORAS:
        hits = sorted(glob.glob('/kaggle/input/' + marker, recursive=True))
        assert hits, f'LoRA checkpoint for {key} not attached ({marker})'
        ck = pathlib.Path(os.path.dirname(hits[0]))
        print(key, ck, flush=True)
        ltok = AutoTokenizer.from_pretrained(ck / 'tokenizer')
        ltok.padding_side = 'right'
        if ltok.pad_token is None:
            ltok.pad_token = ltok.eos_token
        lm = PeftModel.from_pretrained(trunk, ck / 'adapter').eval()
        heads = load_file(str(ck / norms_file))
        norms = torch.nn.ModuleList([torch.nn.LayerNorm(4096) for _ in L])
        norms.load_state_dict({k[len('norms.'):]: v for k, v in heads.items() if k.startswith('norms.')})
        ro = np.load(ck / 'readouts.npz')
        assert list(ro['classes']) == LABELS and list(ro['pool_layers']) == L
        dev = lm.get_input_embeddings().weight.device
        norms = norms.to(dev).float()

        def embed(tl, bs):
            Z = np.zeros((len(tl), 4096 * len(L)), dtype=np.float32)
            order = np.argsort([len(t) for t in tl], kind='stable')
            for b0 in range(0, len(order), bs):
                idx = order[b0:b0 + bs]
                batch = ltok.pad({'input_ids': [tl[i] for i in idx]}, return_tensors='pt').to(dev)
                with torch.autocast('cuda', dtype=torch.float16):
                    hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states
                m = batch['attention_mask'].unsqueeze(-1).float()
                m[:, 0] = 0
                Z[idx] = torch.cat([nm((hs[l].float() * m.to(hs[l].device)).sum(1).to(dev) / m.sum(1).to(dev))
                                    for l, nm in zip(L, norms)], -1).cpu().numpy()
                del hs, batch
            return Z

        def tokens_of(tx):
            return [ids + [ltok.eos_token_id] for ids in
                    ltok(list(tx), add_special_tokens=False, truncation=True, max_length=LORA['max_len'])['input_ids']]

        if (ck / anchors_file).exists():   # the checkpoint saved its features for train rows 0..2
            a = np.load(ck / anchors_file)['features'].astype(np.float32)
            Za = embed(tokens_of(parity_texts), 3)
            rel = float(np.abs(Za - a).mean() / (np.abs(a).mean() + 1e-9))
            print(f'  {key} parity vs saved anchor features: max abs {float(np.abs(Za - a).max()):.4f}, mean rel {rel:.5f}',
                  flush=True)
            assert rel < 0.02, f'{key}: features do not match the checkpoint anchors'
        Z = embed(tokens_of(texts), LORA['bs'])
        lg = Z @ ro['plain_coef'].T + ro['plain_intercept']
        lg -= lg.max(1, keepdims=True)
        p = np.exp(lg)
        add(key, p / p.sum(1, keepdims=True))
        if 'claude_coef' in ro.files:   # within-Claude head (orig Claude rows + public Claude API passages)
            global LC_TE
            lc5 = Z @ ro['claude_coef'].T + ro['claude_intercept']
            lc5 -= lc5.max(1, keepdims=True)
            LC_TE = lc5 - np.log(np.exp(lc5).sum(1, keepdims=True))
        if ADAPT and key in TWIN_W_LORA and ADAPT_ROWS.any():
            Zc = group_center(Z[ADAPT_ROWS], G_TE[ADAPT_ROWS])
            lc = Zc @ ro['center_coef'].T + ro['center_intercept']
            lc -= lc.max(1, keepdims=True)
            LOGIT[ADAPT_ROWS] += TWIN_W_LORA[key] * (lc - np.log(np.exp(lc).sum(1, keepdims=True)))
            print(f'  {key} centered twin on {int(ADAPT_ROWS.sum())} clustered rows', flush=True)
        trunk = lm.unload()   # drop this adapter, keep the trunk for the next checkpoint
        del lm
        gc.collect()
        torch.cuda.empty_cache()
    del trunk
    gc.collect()
    torch.cuda.empty_cache()


if LORA['enabled'] and not SMOKE:
    run_loras(TE, TR[:3])


# ------------------------------------------------ 3d) learned token-attention head on frozen Qwen3.5-9B layer-18 states
class TokAttn(torch.nn.Module):   # same module as the training kernels lucasboesen/name-that-llm-train-tokattn(2)
    def __init__(self, k, h=4096):
        super().__init__()
        self.drop = torch.nn.Dropout(0.2)
        self.proj = torch.nn.Linear(h, 64)
        self.score = torch.nn.Linear(64, 1)
        self.out = torch.nn.Linear(128, k)

    def forward(self, x, m):
        h = torch.tanh(self.proj(self.drop(x)))
        s = self.score(h).squeeze(-1).masked_fill(~m, -1e4)
        a = torch.softmax(s, -1).unsqueeze(-1)
        mf = m.unsqueeze(-1).float()
        mean = (h * mf).sum(1) / mf.sum(1).clamp(min=1)
        return self.out(self.drop(torch.cat([mean, (a * h).sum(1)], -1)))


@torch.no_grad()
def tokattn_proba(texts, head_file, backbone):
    hits = sorted(glob.glob('/kaggle/input/**/' + head_file, recursive=True))
    assert hits, f'attach the training-kernel output holding {head_file}'
    d = torch.load(hits[0], map_location='cpu')
    assert list(d['classes']) == LABELS
    decoder = d.get('decoder', True)
    if backbone == 'qwen3_5':
        path = None
        for c in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
            try:
                if json.load(open(c)).get('model_type') == 'qwen3_5':
                    path = os.path.dirname(c)
                    break
            except Exception:
                continue
    else:
        path = find_dir('deberta-v3-large')
    assert path, f'backbone {backbone} not attached'
    cfg = AutoConfig.from_pretrained(path)
    if decoder:
        tc = getattr(cfg, 'text_config', None) or cfg
        tc.num_hidden_layers = d['keep']
        if getattr(tc, 'layer_types', None):
            tc.layer_types = tc.layer_types[:d['keep']]
        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map='auto').eval()
    else:
        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()
    qtok = AutoTokenizer.from_pretrained(path)
    qtok.padding_side = 'right'
    if qtok.pad_token is None:
        qtok.pad_token = qtok.eos_token
    H = d.get('hidden', 4096)
    net = TokAttn(len(LABELS), H)
    net.load_state_dict(d['state_dict'])
    dev = lm.get_input_embeddings().weight.device
    net = net.to(dev).eval()
    mu, sd = d['mu'].float().to(dev), d['sd'].float().to(dev)
    ids = qtok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d['max_len'])['input_ids']
    P = np.zeros((len(texts), len(LABELS)))
    order = np.argsort([len(t) for t in ids])
    for b0 in range(0, len(order), TOKATTN['bs']):
        idx = order[b0:b0 + TOKATTN['bs']]
        batch = qtok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
        out = lm(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], output_hidden_states=True)
        hs = (out.hidden_states if getattr(out, 'hidden_states', None) is not None
              else out.language_model_outputs.hidden_states)[d['layer']]
        n = batch['attention_mask'].sum(1)
        T = int(n.max()) - 1
        x = torch.zeros(len(idx), max(T, 1), H, device=dev)
        m = torch.zeros(len(idx), max(T, 1), dtype=torch.bool, device=dev)
        for r in range(len(idx)):   # first token dropped, as in training (states stored in fp16 there)
            k = int(n[r]) - 1
            if k > 0:
                st = torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float()
                x[r, :k] = ((st - mu) / sd).clamp(-8, 8)
                m[r, :k] = True
        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()
        del hs, out, batch
    del lm
    gc.collect()
    torch.cuda.empty_cache()
    return P


if TOKATTN['enabled'] and not SMOKE:
    for tkey, tfile, tbackbone in TOKATTN_HEADS:
        if len(TE) < 50:   # interactive run: sanity-check each head on train rows it was fit on (in-sample, must be high)
            pt = tokattn_proba(TR[:40], tfile, tbackbone)
            acc_t = float(np.mean(pt.argmax(1) == y[:40]))
            print(f'  {tkey} in-sample check on 40 train rows: acc {acc_t:.2f}', flush=True)
            assert acc_t > 0.5, f'{tkey} does not reproduce its training fit'
        add(tkey, tokattn_proba(TE, tfile, tbackbone))


# ------------------------------------------------ 4) fine-tuned encoder fold ensembles (softmax averaged over folds)
def find_folds(dir_name, model_type, local_fallbacks):
    """Fold dirs (fold0..fold4) under a Kaggle input dir named dir_name whose config has model_type."""
    roots = glob.glob(f'/kaggle/input/**/{dir_name}', recursive=True) + local_fallbacks
    for r in roots:
        fs = sorted(glob.glob(os.path.join(r, 'fold*')))
        fs = [f for f in fs if os.path.isdir(f) and os.path.exists(os.path.join(f, 'config.json'))
              and json.load(open(os.path.join(f, 'config.json'))).get('model_type') == model_type]
        if len(fs) == 5:
            return fs
    raise FileNotFoundError(f'no 5 {model_type} folds under {dir_name}')


@torch.no_grad()
def encoder_ensemble(folds, remote_code):
    etok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=remote_code)
    enc = etok(TE, truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(e) for e in enc])
    P = np.zeros((len(TE), K))
    for fp in folds:
        em = AutoModelForSequenceClassification.from_pretrained(
            fp, trust_remote_code=remote_code, dtype=torch.float16 if DEV == 'cuda' else torch.float32,
            attn_implementation='sdpa').to(DEV).eval()
        cols = [LABELS.index(em.config.id2label[i]) for i in range(len(em.config.id2label))]
        for i in range(0, len(order), 32):
            idx = order[i:i + 32]
            b = etok.pad({'input_ids': [enc[j] for j in idx]}, return_tensors='pt').to(DEV)
            P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)
        del em
        gc.collect()
        if DEV == 'cuda':
            torch.cuda.empty_cache()
    return P


for key, dir_name, model_type, remote_code, local in ENCODERS:
    add(key, encoder_ensemble(find_folds(dir_name, model_type, local), remote_code))

# ------------------------------------------------ blend + transductive marginal balancing
# Sinkhorn column rescaling of the test posterior toward uniform class mass (alpha = partial strength).
# CV: never negative (in-domain +0.002..+0.007, also with unequal class counts); under genre shift (external
# data) predictions collapsed onto one class and balancing took balanced accuracy from 0.12 to 0.19.
P = np.exp(LOGIT - LOGIT.max(1, keepdims=True))
P /= P.sum(1, keepdims=True)
if BALANCE_ALPHA > 0 and len(P) >= 2:
    target = (1 - BALANCE_ALPHA) * P.sum(0) + BALANCE_ALPHA * np.full(K, len(P) / K)
    for _ in range(50):
        P *= (target / (P.sum(0) + 1e-12))[None, :]
        P /= P.sum(1, keepdims=True) + 1e-12
    print('balancing changed', int((P.argmax(1) != LOGIT.argmax(1)).sum()), 'of', len(P), flush=True)
pred = P.argmax(1)
if DC_TE is not None:
    _isc = np.isin(pred, CLAUDE_IDX)
    _s = np.log(np.clip(P[:, CLAUDE_IDX], 1e-12, 1)) + CLAUDE_GATE_LAM * DC_TE
    if LC_TE is not None:
        _s = _s + CLAUDE_LORA_LAM * LC_TE
    _new = CLAUDE_IDX[_s.argmax(1)]
    print('claude gate changed', int((_new[_isc] != pred[_isc]).sum()), 'of', int(_isc.sum()), 'Claude predictions', flush=True)
    pred[_isc] = _new[_isc]
sub = pd.DataFrame({'id': test['id'], 'model': [LABELS[i] for i in pred]})
assert sub['id'].is_unique and sub['model'].isin(LABELS).all() and len(sub) == len(test)
sub.to_parquet('submission.parquet', index=False)
print(sub['model'].value_counts().to_string())
print(f'total {time.time() - T0:.0f}s')

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: capture_runtime.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('capture_runtime.py', r'''
"""Capture existing frozen features and balanced baseline into ephemeral storage."""
import hashlib
import json
import os
from functools import wraps
from pathlib import Path
import numpy as np
from claude_gate import save_gate_state

VIEWS = {'q3_8b_cat7+22':8192, 'style4b':5120}

def cache_directory():
    directory = Path(os.environ['NTL_TABULAR_TEMP']).resolve()
    assert directory.is_relative_to(Path('/kaggle/temp').resolve())
    directory.mkdir(parents=True,exist_ok=True)
    return directory

def wrap_fit_head(original, train, test, labels):
    cache = cache_directory()
    np.savez_compressed(cache/'alignment.npz',
        train_ids=train.id.to_numpy(dtype=str), test_ids=test.id.to_numpy(dtype=str),
        labels=np.asarray(labels,dtype=str),
        y=train.model.map({label:i for i,label in enumerate(labels)}).to_numpy(dtype=np.int64))
    captured = set()
    @wraps(original)
    def fitting(key, Etr, Ete, C):
        if key in VIEWS:
            assert key not in captured
            assert Etr.dtype==Ete.dtype==np.float32
            assert Etr.shape==(len(train),VIEWS[key]) and Ete.shape==(len(test),VIEWS[key])
            assert np.isfinite(Etr).all() and np.isfinite(Ete).all()
            for suffix,value in [('train',Etr),('test',Ete)]:
                np.save(cache/(key+'_'+suffix+'.npy'),value,allow_pickle=False)
            captured.add(key)
            (cache/'captured_views.json').write_text(json.dumps(sorted(captured)))
        return original(key,Etr,Ete,C)
    return fitting

def capture_balanced(P,test,labels,DC_TE,LC_TE,CLAUDE_IDX,gate_lam,lora_lam):
    cache=cache_directory()
    assert set(json.loads((cache/'captured_views.json').read_text()))==set(VIEWS)
    assert P.dtype==np.float64 and P.shape==(len(test),len(labels))
    assert np.isfinite(P).all() and (P>=0).all()
    np.savez_compressed(cache/'balanced_baseline.npz',probabilities=P,
        ids=test.id.to_numpy(dtype=str),labels=np.asarray(labels,dtype=str))
    save_gate_state(cache,P,test.id.to_numpy(dtype=str),labels,
        DC_TE,LC_TE,CLAUDE_IDX,gate_lam,lora_lam)
    records={}
    for path in sorted(cache.glob('*.npy')):
        with path.open('rb') as stream: digest=hashlib.file_digest(stream,'sha256').hexdigest()
        records[path.name]=dict(bytes=path.stat().st_size,sha256=digest)
    # Only sizes and hashes are exportable; arrays and targets stay under /kaggle/temp.
    Path('/kaggle/working/tabular_capture_audit.json').write_text(json.dumps(dict(
        status='TWO_EXISTING_VIEWS_CAPTURED_WITHOUT_EXTRA_BACKBONE_PASS',
        train_rows=len(np.load(cache/'alignment.npz',allow_pickle=False)['train_ids']),
        query_rows=len(test),views=records,baseline_classes=list(labels),
        source='exact ntl-v6 v2 raw fit_head inputs before preprocessing',
        gate_state_captured_before_original_argmax=True,
        dc_dtype=None if DC_TE is None else str(DC_TE.dtype),
        lc_dtype=None if LC_TE is None else str(LC_TE.dtype),
        claude_indices_dtype=str(CLAUDE_IDX.dtype),
        ephemeral_training_arrays=True),indent=2))

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: claude_gate.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('claude_gate.py', r'''
"""Exact ntl-v6 hard Claude decision, with lossless temporary state transport."""
from pathlib import Path
import numpy as np


def validate_gate(P, labels, DC_TE, LC_TE, CLAUDE_IDX, gate_lam, lora_lam):
    labels = np.asarray(labels)
    assert labels.ndim == 1 and len(labels) == len(np.unique(labels)) == 18
    assert P.dtype == np.float64 and P.shape[1:] == (len(labels),)
    assert np.isfinite(P).all() and (P >= 0).all()
    expected = np.array([i for i, label in enumerate(labels) if label.startswith('claude')])
    assert len(expected) == 5
    assert np.issubdtype(CLAUDE_IDX.dtype, np.integer)
    np.testing.assert_array_equal(CLAUDE_IDX, expected)
    assert (gate_lam, lora_lam) == (1.5, 1.0)
    for value in (DC_TE, LC_TE):
        if value is not None:
            assert np.issubdtype(value.dtype, np.floating)
            assert value.shape == (len(P), len(CLAUDE_IDX)) and np.isfinite(value).all()


def gated_predict(P, labels, DC_TE, LC_TE, CLAUDE_IDX, gate_lam=1.5, lora_lam=1.0):
    validate_gate(P, labels, DC_TE, LC_TE, CLAUDE_IDX, gate_lam, lora_lam)
    # Keep the reference operations, dtypes, clipping, tie order and optional gate exactly.
    pred = P.argmax(1)
    if DC_TE is not None:
        _isc = np.isin(pred, CLAUDE_IDX)
        _s = np.log(np.clip(P[:, CLAUDE_IDX], 1e-12, 1)) + gate_lam * DC_TE
        if LC_TE is not None:
            _s = _s + lora_lam * LC_TE
        _new = CLAUDE_IDX[_s.argmax(1)]
        pred[_isc] = _new[_isc]
    return pred


def save_gate_state(directory, P, ids, labels, DC_TE, LC_TE, CLAUDE_IDX, gate_lam, lora_lam):
    validate_gate(P, labels, DC_TE, LC_TE, CLAUDE_IDX, gate_lam, lora_lam)
    # Explicit flags avoid object arrays and preserve the actual present-array dtypes.
    np.savez_compressed(Path(directory)/'claude_gate_state.npz',
        ids=np.asarray(ids, dtype=str), labels=np.asarray(labels, dtype=str),
        dc_present=np.asarray(DC_TE is not None), lc_present=np.asarray(LC_TE is not None),
        DC_TE=DC_TE if DC_TE is not None else np.empty((0, 5), dtype=np.float64),
        LC_TE=LC_TE if LC_TE is not None else np.empty((0, 5), dtype=np.float64),
        CLAUDE_IDX=CLAUDE_IDX, gate_lam=np.asarray(gate_lam), lora_lam=np.asarray(lora_lam))


def load_gate_state(directory, P, ids, labels):
    with np.load(Path(directory)/'claude_gate_state.npz', allow_pickle=False) as saved:
        np.testing.assert_array_equal(saved['ids'], ids)
        np.testing.assert_array_equal(saved['labels'], labels)
        state = dict(DC_TE=saved['DC_TE'].copy() if bool(saved['dc_present']) else None,
            LC_TE=saved['LC_TE'].copy() if bool(saved['lc_present']) else None,
            CLAUDE_IDX=saved['CLAUDE_IDX'].copy(),
            gate_lam=saved['gate_lam'].item(), lora_lam=saved['lora_lam'].item())
    validate_gate(P, labels, **state)
    return state

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: masked_features.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('masked_features.py', r'''
"""Fixed shape/function TF-IDF-SVD blocks; all learned transforms fit on train only."""

import re

import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.decomposition import TruncatedSVD

FUNCTION_WORDS_TEXT='a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'

def shape_text(text):
    # Preserve character count/case, punctuation and whitespace; suppress letters/digits.
    return re.sub(r'\d','#',re.sub(r'[^\W\d_]',lambda m:'A' if m[0].isupper() else 'a',text))

def function_text(text):
    words=set(FUNCTION_WORDS_TEXT.split())
    return ' '.join(t.lower() if t.lower() in words else ('CONTENT' if t.isalnum() else t)
        for t in re.findall(r'\w+|[^\w\s]',text))

VECTORIZERS = {'shape_char': {'min_df': 3, 'max_df': 0.995, 'sublinear_tf': True, 'norm': 'l2', 'analyzer': 'char', 'ngram_range': [3, 5], 'lowercase': False, 'max_features': 20000}, 'function_word': {'min_df': 3, 'max_df': 0.995, 'sublinear_tf': True, 'norm': 'l2', 'analyzer': 'word', 'ngram_range': [1, 3], 'lowercase': False, 'token_pattern': '(?u)\\b\\w+\\b|[^\\w\\s]', 'max_features': 20000}}

def masked_svd_blocks(train_texts, query_texts):
    train_blocks, query_blocks, audit = [], [], []
    for name, cfg in VECTORIZERS.items():
        masker = shape_text if name == 'shape_char' else function_text
        fit_values = [masker(text) for text in train_texts]
        query_values = [masker(text) for text in query_texts]
        options = dict(cfg)
        options['ngram_range'] = tuple(options['ngram_range'])
        options['dtype'] = np.float32
        vectorizer = TfidfVectorizer(**options)
        sparse_fit = vectorizer.fit_transform(fit_values)
        assert sparse_fit.shape[1] > 16
        svd = TruncatedSVD(n_components=16, n_iter=5, random_state=20261017)
        svd.fit(sparse_fit)
        train_block = svd.transform(vectorizer.transform(fit_values)).astype(np.float32)
        query_block = svd.transform(vectorizer.transform(query_values)).astype(np.float32)
        assert train_block.shape == (len(train_texts), 16)
        assert query_block.shape == (len(query_texts), 16)
        assert np.isfinite(train_block).all() and np.isfinite(query_block).all()
        train_blocks.append(train_block)
        query_blocks.append(query_block)
        audit.append(dict(view=name, vocabulary_size=len(vectorizer.vocabulary_),
            fit_rows=len(train_texts), query_rows=len(query_texts), components=16,
            fit_scope='official_train_only',
            explained_variance_ratio_sum=float(svd.explained_variance_ratio_.sum())))
    return np.concatenate(train_blocks, axis=1), np.concatenate(query_blocks, axis=1), audit


'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: tabular_worker.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('tabular_worker.py', r'''
"""Fixed train-only feature preparation or one isolated tabular inference member."""
import hashlib
import importlib.metadata
import json
import os
import re
import sys
import time
from pathlib import Path

assert os.name!='nt' and Path('/kaggle/input').is_dir()
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):os.environ[name]='4'
os.environ['HF_HUB_OFFLINE']='1'
import numpy as np

here=Path(__file__).resolve().parent
config=json.loads((here/'runtime_config.json').read_text())
cache=Path(os.environ['NTL_TABULAR_TEMP']).resolve()
assert cache.is_relative_to(Path('/kaggle/temp').resolve())
mode=sys.argv[1]
started=time.monotonic()

def sha(path):
    with path.open('rb') as stream:return hashlib.file_digest(stream,'sha256').hexdigest()

def style24(t):
    # Original reference first 20 deterministic scalars, plus four fixed ratios.
    words = re.findall(r"[A-Za-z\u2019']+", t)
    nw, nc = max(len(words), 1), max(len(t), 1)
    sents = [x for x in re.split(r'(?<=[.!?])\s+', t.strip()) if x]
    sl = np.array([len(re.findall(r"[A-Za-z\u2019']+", x)) for x in sents]) if sents else np.array([0])
    wl = np.array([len(x) for x in words]) if words else np.array([0])
    lw = [x.lower() for x in words]
    return [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(),
        (wl >= 9).mean(), len(set(lw))/nw, sum(c.isupper() for c in t)/nc, sum(c.isdigit() for c in t)/nc,
        sum(ord(c)>127 for c in t)/nc, t.count(', ')/nw, t.count(' and ')/nw, t.count(' but ')/nw,
        sum(x[:1].isupper() and x.split()[0].lower() in ('this','these','it','they') for x in sents)/max(len(sents),1),
        sum(x.endswith('ly') for x in words)/nw, sum(x.endswith('ing') for x in words)/nw,
        sum(x.endswith('tion') for x in words)/nw,
        float(any(c in t for c in ('\u2019','\u201c','\u201d'))), t.count('\u2014')/nw,
        t.count('\n')/nc, sum(bool(x.strip()) for x in t.splitlines())/nw]

with np.load(cache/'alignment.npz',allow_pickle=False) as saved:
    alignment={key:saved[key].copy() for key in saved.files}
if mode=='prepare':
    import pandas as pd
    from sklearn.decomposition import PCA
    from threadpoolctl import threadpool_limits
    threadpool_limits(4)
    data=next(path for path in [Path('/kaggle/input/competitions/name-that-llm'),Path('/kaggle/input/name-that-llm')] if (path/'train.csv').is_file())
    train,test=pd.read_csv(data/'train.csv'),pd.read_csv(data/'test.csv')
    for frame,key in [(train,'train_ids'),(test,'test_ids')]:
        np.testing.assert_array_equal(frame.id.to_numpy(dtype=str),alignment[key])
    np.testing.assert_array_equal(train.model.to_numpy(dtype=str),alignment['labels'][alignment['y']])
    train_columns,test_columns,audit=[],[],[]
    for key,width in [('q3_8b_cat7+22',8192),('style4b',5120)]:
        train_view=np.load(cache/(key+'_train.npy'),mmap_mode='r',allow_pickle=False)
        test_view=np.load(cache/(key+'_test.npy'),mmap_mode='r',allow_pickle=False)
        assert train_view.shape==(len(train),width) and test_view.shape==(len(test),width)
        pca=PCA(n_components=64,svd_solver='randomized',iterated_power=3,random_state=42)
        pca.fit(train_view)
        train_columns.append(pca.transform(train_view).astype(np.float32))
        test_columns.append(pca.transform(test_view).astype(np.float32))
        audit.append(dict(view=key,fit_rows=len(train),components=64,fit_scope='official_train_only',
            explained_variance=float(pca.explained_variance_ratio_.sum())))
    for frame,columns in [(train,train_columns),(test,test_columns)]:
        texts=frame.text.fillna('').astype(str)
        columns.append(np.asarray([style24(text) for text in texts],dtype=np.float32))
    assert sha(here/'masked_features.py')==config['masked_feature_module_sha256']
    base_train,base_test=np.concatenate(train_columns,axis=1),np.concatenate(test_columns,axis=1)
    from masked_features import masked_svd_blocks
    text_train,text_test,text_audit=masked_svd_blocks(
        train.text.fillna('').astype(str).tolist(),test.text.fillna('').astype(str).tolist())
    train_columns.append(text_train)
    test_columns.append(text_test)
    x_train,x_test=np.concatenate(train_columns,axis=1),np.concatenate(test_columns,axis=1)
    assert x_train.shape==(len(train),184) and x_test.shape==(len(test),184)
    np.testing.assert_array_equal(x_train[:,:152],base_train)
    np.testing.assert_array_equal(x_test[:,:152],base_test)
    assert np.isfinite(x_train).all() and np.isfinite(x_test).all()
    np.savez(cache/'tables.npz',x_train=x_train,x_test=x_test,**alignment)
    Path('/kaggle/working/tabular_prepare_audit.json').write_text(json.dumps(dict(
        status='FULL_TRAIN_ONLY_PCA_STYLE24_AND_MASKED_SVD_COMPLETE',pca=audit,text_svd=text_audit,features=184,
        masked_feature_module_sha256=config['masked_feature_module_sha256'],
        query_rows=len(test),elapsed_seconds=time.monotonic()-started,
        packages={name:importlib.metadata.version(name) for name in ('numpy','pandas','scikit-learn','scipy')},
        training_context_exported=False),indent=2))
    sys.exit(0)

assert mode=='tabpfn'
import torch
assert torch.cuda.is_available()
assert torch.cuda.mem_get_info()[0]>8*1024**3
with np.load(cache/'tables.npz',allow_pickle=False) as saved:
    x_train,x_test,y=(saved[key].copy() for key in ('x_train','x_test','y'))
asset=Path(os.environ['NTL_TABULAR_ASSET'])
settings=config[mode]
checkpoint=asset/settings['checkpoint_filename']
assert checkpoint.is_file() and sha(checkpoint)==settings['checkpoint_sha256']
assert importlib.metadata.version(mode)==settings['package_version']
assert settings['n_estimators']==128,'Preserve the selected128-estimator configuration'
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
params=dict(n_estimators=128,model_path=str(checkpoint),device='cuda:0',fit_mode='low_memory',
    memory_saving_mode='auto',balance_probabilities=True,tuning_config=None,random_state=42,n_preprocessing_jobs=1)
model=TabPFNClassifier.create_default_for_version(ModelVersion.V3_5,**params)
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize();t=time.monotonic()
model.fit(x_train,y)
torch.cuda.synchronize();fit_seconds=time.monotonic()-t
t=time.monotonic();raw=model.predict_proba(x_test)
torch.cuda.synchronize();predict_seconds=time.monotonic()-t
classes=np.asarray(model.classes_,dtype=int)
np.testing.assert_array_equal(np.sort(classes),np.arange(18))
probabilities=np.zeros((len(x_test),18),dtype=np.float64)
probabilities[:,classes]=raw
assert np.isfinite(probabilities).all() and (probabilities>=0).all()
np.testing.assert_allclose(probabilities.sum(1),1,rtol=0,atol=1e-5)
np.savez(cache/(mode+'_proba.npz'),proba=probabilities,ids=alignment['test_ids'],labels=alignment['labels'])
Path('/kaggle/working/'+mode+'_inference_audit.json').write_text(json.dumps(dict(
    status='COMPLETE',model=mode,checkpoint_sha256=settings['checkpoint_sha256'],parameters=params,
    train_rows=len(x_train),query_rows=len(x_test),features=184,fit_seconds=fit_seconds,
    predict_seconds=predict_seconds,peak_cuda_allocated_bytes=int(torch.cuda.max_memory_allocated()),
    peak_cuda_reserved_bytes=int(torch.cuda.max_memory_reserved()),probability_transform='none',
    row_mass_maxabs=float(np.max(np.abs(probabilities.sum(1)-1))),
    packages={name:importlib.metadata.version(name) for name in (mode,'torch','numpy','scikit-learn')},
    oof=False,model_context_exported=False),indent=2))

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: run_baseline.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('run_baseline.py', r'''
"""Exact frozen B556 with one reversible observation hook and post-execution exports."""
import hashlib
import json
import os
from pathlib import Path
from capture_runtime import wrap_fit_head, capture_balanced

assert os.name != 'nt' and Path('/kaggle/input').is_dir()
here = Path(__file__).resolve().parent
config = json.loads((here / 'runtime_config.json').read_text())
raw = (here / 'ntl_reference.py').read_bytes()
assert hashlib.sha256(raw).hexdigest() == config['reference_source_sha256']
source = raw.decode('utf-8')
assert config['baseline_balance_alpha'] == .5
assert config['reference_notebook_id'] == 139200100 and config['reference_version'] == 2
assert 'BALANCE_ALPHA = 0.5' in source and "W = {'stylegram': 1.516" in source
loop = 'for key, mt, hid, layers, C, smoke_id, pool in LLM_HEADS:'
hook = 'fit_head = _tabular_wrap_fit_head(fit_head, train, test, LABELS)\n' + loop
assert source.count(loop) == 1
modified = source.replace(loop, hook)
assert modified.replace(hook, loop) == source
namespace = dict(__name__='__main__', __file__=str(here / 'ntl_reference.py'),
                 _tabular_wrap_fit_head=wrap_fit_head)
exec(compile(modified, 'B556_observation_only', 'exec'), namespace)
assert Path('/kaggle/working/submission.parquet').is_file()
import numpy as np
cache = Path(os.environ['NTL_TABULAR_TEMP']).resolve()
assert cache.is_relative_to(Path('/kaggle/temp').resolve())
P, raw_logits, test, labels = (namespace[key] for key in ('P', 'LOGIT', 'test', 'LABELS'))
groups, adaptive, pred = (namespace[key] for key in ('G_TE', 'ADAPT_ROWS', 'pred'))
assert P.dtype == raw_logits.dtype == np.float64 and P.shape == raw_logits.shape == (len(test), 18)
assert groups.shape == adaptive.shape == pred.shape == (len(test),)
assert np.issubdtype(groups.dtype, np.integer) and adaptive.dtype == bool
sizes = np.bincount(groups)[groups]
np.testing.assert_array_equal(adaptive, (sizes >= 2) & (sizes <= 18))
# Original code has already finished. Extra I/O observes values without rerunning its gate or models.
capture_balanced(P, test, labels, namespace['DC_TE'], namespace['LC_TE'], namespace['CLAUDE_IDX'],
                 namespace['CLAUDE_GATE_LAM'], namespace['CLAUDE_LORA_LAM'])
np.savez_compressed(cache / 'baseline_observation.npz', ids=test.id.to_numpy(dtype=str),
    labels=np.asarray(labels, dtype=str), raw=raw_logits, balanced=P, gated_predictions=pred,
    recovered_group_ids=groups, adaptive_mask=adaptive, pool_ids=np.zeros(len(test), dtype=np.int64))
Path('/kaggle/working/B556_source_parity.json').write_text(json.dumps(dict(
    source_notebook_id=139200100, source_version=2, source_script_version_id=359313348,
    original_source_sha256=hashlib.sha256(raw).hexdigest(), executed_source_sha256=hashlib.sha256(modified.encode()).hexdigest(),
    observation_hook_reversible=True, original_W_TWIN_and_all_scientific_bodies_unchanged=True,
    original_balance_alpha=.5, original_stylegram_lambda=1, original_gate_last=True,
    exported_values=['unbalanced_raw', 'balanced_P', 'original_gated_labels', 'actual_recovered_groups', 'actual_adaptive_mask'],
    query_pool_semantics='One actual complete test CSV pool; pool_id0; source G_TE is reused without reclustering',
    observation_arrays_saved_under_temp_only=True, extra_language_model_passes=0), indent=2))

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: protected_operator.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('protected_operator.py', r'''
import numpy as np

def protected(q,p,baseline,claude,nonclaude,weight=.10):
    protected = np.isin(baseline, claude)
    np.testing.assert_array_equal(protected, np.isin(q.argmax(-1), claude))
    candidate = q.copy()
    reverted = np.zeros(q.shape[:2], dtype=bool)
    for context in range(q.shape[0]):
        rows = np.flatnonzero(~protected[context])
        original_non = q[context][np.ix_(rows, nonclaude)]
        non_mass = original_non.sum(1, keepdims=True)
        p_non = p[np.ix_(rows, nonclaude)]
        assert (non_mass > 0).all() and (p_non.sum(1) > 0).all()
        conditional = np.power(original_non / non_mass, 1 - weight) * np.power(p_non / p_non.sum(1, keepdims=True), weight)
        assert np.isfinite(conditional).all() and (conditional.sum(1) > 0).all()
        conditional /= conditional.sum(1, keepdims=True)
        candidate[context][np.ix_(rows, nonclaude)] = non_mass * conditional
        crossed = np.isin(candidate[context, rows].argmax(-1), claude)
        reverted[context, rows[crossed]] = True
        candidate[context, rows[crossed]] = q[context, rows[crossed]]
    # Retain original expert-selected Claude labels; only eligible non-Claude argmax labels may change.
    prediction = baseline.copy()
    prediction[~protected] = candidate.argmax(-1)[~protected]
    assert candidate.dtype == np.float64 and np.isfinite(candidate).all() and (candidate >= 0).all()
    assert candidate[:, :, claude].tobytes() == q[:, :, claude].tobytes(), "Claude columns changed at the bit level"
    assert candidate[protected].tobytes() == q[protected].tobytes(), "Protected whole row changed"
    assert candidate[reverted].tobytes() == q[reverted].tobytes(), "Family flip was not fully reverted"
    np.testing.assert_array_equal(prediction[protected], baseline[protected])
    np.testing.assert_array_equal(np.isin(prediction, claude), protected)
    np.testing.assert_allclose(candidate[:, :, nonclaude].sum(-1), q[:, :, nonclaude].sum(-1), rtol=0, atol=1e-15)
    np.testing.assert_allclose(candidate.sum(-1), q.sum(-1), rtol=0, atol=1e-15)
    np.testing.assert_allclose(candidate.sum(-1), 1, rtol=0, atol=2e-12)
    return candidate,prediction,reverted

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: QWEN35_LICENSE
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('QWEN35_LICENSE', r'''

                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright 2026 Alibaba Cloud

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.
'''[1:-1], crlf=True)


In [ ]:
'''Name That LLM | Upstream Module: QWEN35_NOTICE.md
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('QWEN35_NOTICE.md', r'''
Qwen3.5-9B
===========

Copyright 2026 Alibaba Cloud. Licensed under Apache License 2.0; see QWEN35_LICENSE.

Official repository: https://huggingface.co/Qwen/Qwen3.5-9B
License source: https://huggingface.co/Qwen/Qwen3.5-9B/blob/c202236235762e1c871ad0ccb60c8ee5ba337b9a/LICENSE

The existing Kaggle input bbucxi/qwen35-9b-safetensors/1 supplies the model. Its config and tensor index match the official repository at the revision above byte-for-byte; all four weight shard names and byte lengths agree. This attribution records the upstream license and does not change the input weights.

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: component.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('component.py', r'''
"""Disabled-by-default portable Smol final plain component; no training, targets, or optimizer."""
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path


def sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def validate_contract(spec, artifact_root, base_root, diagnostic_only=False):
    """Stdlib-only full-final admission and immutable portable file checks."""
    assert spec["actual_final_binding"], "Source only: actual full-final QA/binding missing"
    binding = spec["actual_final_binding"]
    assert binding["actual_full_QA_passed"] is True
    if diagnostic_only:
        assert spec["diagnostic_only_authorized"] is True and spec["inference_authorized"] is False
        assert binding["paired_gain_admitted_by_root"] is False, "Diagnostic must not claim candidate admission"
    else:
        assert spec["inference_authorized"] is True and binding["paired_gain_admitted_by_root"] is True
    assert binding["notebook_id"] == 139347470 and binding["version"] == 1
    assert binding["script_version_id"] > 0
    assert binding["worker_sha256"] == spec["training_worker_sha256"]
    assert binding["checkpoint_relative"] == "final/finished_checkpoint"
    artifact_root, base_root = Path(artifact_root), Path(base_root)
    complete_path = artifact_root / "final/COMPLETE.json"
    report_path = artifact_root / "final/continuation-report.json"
    assert sha(complete_path) == binding["complete_sha256"] and sha(report_path) == binding["report_sha256"]
    complete, report = json.loads(complete_path.read_text()), json.loads(report_path.read_text())
    assert complete["report_sha256"] == binding["report_sha256"]
    assert complete["status"] == report["status"] == "COMPLETE_SMOL_EXACT_CONTINUATION"
    assert complete["job"] == report["job"] == "final" and complete["successful_updates"] == report["successful_updates"] == 450
    assert report["full_planned_updates"] == 450 and report["resumed_successful_updates"] == 20
    assert report["new_successful_updates"] == 430 and report["completed_epochs"] == report["full_planned_epochs"] == 3
    assert report["final_is_not_OOF"] and report["fit_rows"] == list(range(2699)) and report["valid_rows"] == []
    assert len(report["fit_ids"]) == len(set(report["fit_ids"])) == 2699 and report["valid_ids"] == []
    assert report["classes"] == spec["classes"] and report["train_sha256"] == spec["train_sha256"]
    assert report["worker_sha256"] == spec["training_worker_sha256"]
    assert report["base_revision"] == spec["base_revision"] and report["base_manifest_sha256"] == spec["base_manifest_sha256"]
    assert report["versions"] == spec["core_versions"] and report["pretrained_base_only"] and not report["quantization"]
    assert report["strict_reload_replay_passed"] and report["inference_artifact_ready"] and not report["validation_scored"]
    assert complete["inference_artifacts_ready"] and report["checkpoint"] == "finished_checkpoint"
    assert report["query_artifact"] is None and complete["query_artifact"] is None
    assert max(report["probability_replay_maxabs"].values()) <= 1e-6
    assert [record["update"] for record in report["updates"]] == list(range(1, 451))
    assert all(record["finite_gradients"] for record in report["updates"])
    checkpoint = artifact_root / binding["checkpoint_relative"]
    required = {"adapter/adapter_model.safetensors", "adapter/adapter_config.json", "heads.safetensors",
                "readouts.npz", "replay_anchors.npz"}
    assert required <= set(binding["portable_files_sha256"])
    for relative, digest in binding["portable_files_sha256"].items():
        path = checkpoint / relative
        assert path.resolve().is_relative_to(checkpoint.resolve())
        assert relative != "continuation_state.pt", "Optimizer state is unnecessary for inference"
        assert digest == complete["checkpoint_files_sha256"][relative] and sha(path) == digest
    config = json.loads((checkpoint / "adapter/adapter_config.json").read_text())
    assert config["r"] == 16 and config["lora_alpha"] == 32 and config["lora_dropout"] == .05
    assert config["bias"] == "none" and config["peft_type"] == "LORA"
    stage_path = base_root / "smollm3_staging_complete.json"
    assert sha(stage_path) == spec["base_manifest_sha256"]
    stage = json.loads(stage_path.read_text())
    assert stage["status"] == "COMPLETE_PINNED_PUBLIC_BASE_ONLY" and stage["revision"] == spec["base_revision"]
    assert stage["license"] == "Apache-2.0" and not stage["competition_data"] and not stage["model_initialized"]
    base_dir = base_root / "smollm3-base"
    assert len(stage["files"]) == 9
    for record in stage["files"]:
        assert sha(base_dir / record["name"]) == record["sha256"]
        assert (base_dir / record["name"]).stat().st_size == spec["base_files"][record["name"]]["size"]
        pinned = spec["base_files"][record["name"]].get("lfs_sha256")
        assert not pinned or pinned == record["sha256"]
    return checkpoint, base_dir, report


class SmolFinalPlain:
    def __init__(self, spec_path, artifact_root, base_root, anchor_train_csv, diagnostic_only=False):
        self.spec = json.loads(Path(spec_path).read_text())
        for name, digest in self.spec["source_hashes"].items():
            assert sha(Path(__file__).parent / name) == digest
        checkpoint, base_dir, report = validate_contract(self.spec, artifact_root, base_root, diagnostic_only=diagnostic_only)
        assert os.name != "nt" and Path("/kaggle/input").is_dir(), "Remote-only component"
        assert sha(anchor_train_csv) == self.spec["train_sha256"]
        os.environ["HF_HUB_OFFLINE"] = "1"
        os.environ["TRANSFORMERS_OFFLINE"] = "1"
        os.environ["TOKENIZERS_PARALLELISM"] = "false"
        import importlib.util
        finder = importlib.util.find_spec
        importlib.util.find_spec = lambda name, *a, **kw: None if name.split(".")[0] == "torchao" else finder(name, *a, **kw)
        # Keep the successful FULL worker's torchao mask active through PEFT dispatch.
        import torch
        from torch import nn
        from peft import PeftModel
        import peft.import_utils as peft_imports
        peft_imports.is_torchao_available = lambda *a, **kw: False
        import numpy as np
        import pandas as pd
        from safetensors.torch import load_file
        from scipy.special import log_softmax, softmax
        from owned_path import load_owned_base, make_representation, tokenize_owned

        assert torch.cuda.is_available() and torch.cuda.get_device_capability(0) == (7, 5)
        assert {name: importlib.metadata.version(name) for name in self.spec["core_versions"]} == self.spec["core_versions"]
        torch.set_num_threads(2)
        loading = []
        base = load_owned_base(base_dir, loading, lambda: None)
        self.model = PeftModel.from_pretrained(base, checkpoint / "adapter", is_trainable=False, local_files_only=True).eval()
        assert not any(parameter.requires_grad for parameter in self.model.parameters())
        assert all(parameter.dtype == torch.float32 for name, parameter in self.model.named_parameters() if "lora_" in name)
        self.layers = [6, 12, 18, 24, 30, 36]
        self.norms = nn.ModuleList([nn.LayerNorm(2048) for _ in self.layers]).cuda().float()
        saved = load_file(checkpoint / "heads.safetensors")
        assert all(value.dtype == torch.float32 and bool(torch.isfinite(value).all()) for value in saved.values())
        self.norms.load_state_dict({key.removeprefix("norms."): value for key, value in saved.items() if key.startswith("norms.")}, strict=True)
        self.norms.eval()
        self.norms.requires_grad_(False)
        with np.load(checkpoint / "readouts.npz", allow_pickle=False) as readouts:
            self.readouts = {key: readouts[key].copy() for key in readouts.files}
        np.testing.assert_array_equal(self.readouts["classes"], np.asarray(self.spec["classes"], dtype=str))
        np.testing.assert_array_equal(self.readouts["pool_layers"], self.layers)
        assert self.readouts["plain_coef"].shape == self.readouts["center_coef"].shape == (18, 12288)
        assert self.readouts["plain_intercept"].shape == self.readouts["center_intercept"].shape == (18,)
        for route, prefix in (("plain", "plain"), ("center", "centered")):
            for key, state_key in (("coef", "weight"), ("intercept", "bias")):
                array = self.readouts[route + "_" + key]
                assert array.dtype == np.float32 and np.isfinite(array).all()
                np.testing.assert_array_equal(array, saved[prefix + "." + state_key].numpy())
        # Read only anchor IDs/text, never a target/model column. B3 and padding match the saved replay.
        anchor_data = pd.read_csv(anchor_train_csv, usecols=["id", "text"], dtype=str)
        assert len(anchor_data) == 2699 and anchor_data.id.tolist() == report["fit_ids"]
        self._diagnostic_only = diagnostic_only
        self._diagnostic_ids = anchor_data.id.iloc[:5].tolist()
        self._diagnostic_texts = anchor_data.text.iloc[:5].tolist()
        self.tokenizer, tokens = tokenize_owned(base_dir, anchor_data.text.iloc[:3].tolist())
        self._torch, self._np = torch, np
        self._make_representation, self._tokenize_owned = make_representation, tokenize_owned
        self._log_softmax = log_softmax
        with np.load(checkpoint / "replay_anchors.npz", allow_pickle=False) as anchors:
            np.testing.assert_array_equal(anchors["rows"], [0, 1, 2])
            np.testing.assert_array_equal(anchors["ids"], anchor_data.id.iloc[:3].to_numpy(dtype=str))
            np.testing.assert_array_equal(anchors["classes"], self.readouts["classes"])
            expected, recorded_reloaded = anchors["features"].copy(), anchors["reloaded_features"].copy()
        fresh = self._features(anchor_data.text.iloc[:3].tolist(), sort_by_length=False)
        assert expected.dtype == fresh.dtype == np.float32 and np.isfinite(expected).all()
        np.testing.assert_allclose(fresh, expected, rtol=1e-5, atol=1e-5)
        np.testing.assert_allclose(fresh, recorded_reloaded, rtol=1e-5, atol=1e-5)
        parity = {}
        for route in ("plain", "center"):
            a = expected if route == "plain" else expected - expected.mean(0)
            b = fresh if route == "plain" else fresh - fresh.mean(0)
            coef, intercept = self.readouts[route + "_coef"], self.readouts[route + "_intercept"]
            before = softmax(log_softmax(a @ coef.T + intercept, axis=1), axis=1)
            after = softmax(log_softmax(b @ coef.T + intercept, axis=1), axis=1)
            parity[route] = float(np.max(np.abs(before - after)))
            assert parity[route] <= 1e-6
        self.anchor_replay = {"feature_maxabs": float(np.max(np.abs(fresh - expected))), "probability_maxabs": parity,
                              "feature_rtol": 1e-5, "feature_atol": 1e-5, "probability_atol": 1e-6}

    def _features(self, texts, sort_by_length=True):
        torch, np = self._torch, self._np
        tokenizer, tokens = self._tokenize_owned(None, texts, tokenizer=self.tokenizer)
        representation = self._make_representation(self.model, tokenizer, tokens, self.layers, self.norms)
        features = np.empty((len(tokens), 12288), dtype=np.float32)
        order = np.argsort(np.asarray([len(row) for row in tokens]), kind="stable") if sort_by_length else np.arange(len(tokens))
        with torch.inference_mode():
            for start in range(0, len(order), 3):
                positions = order[start:start + 3]
                features[positions] = representation(positions)[0].cpu().numpy()
        assert np.isfinite(features).all()
        return features

    def predict_plain_logp(self, ids, texts):
        np = self._np
        ids = np.asarray(ids, dtype=str)
        assert ids.ndim == 1 and len(ids) == len(texts) > 0 and len(np.unique(ids)) == len(ids)
        if self._diagnostic_only:
            assert len(ids) in (3, 5) and ids.tolist() == self._diagnostic_ids[:len(ids)]
            assert list(texts) == self._diagnostic_texts[:len(ids)], "Only exact owned3/5-row diagnostic fixtures admitted"
        features = self._features(texts)
        logp = self._log_softmax(features @ self.readouts["plain_coef"].T + self.readouts["plain_intercept"], axis=1)
        assert logp.shape == (len(ids), 18) and logp.dtype == np.float32 and np.isfinite(logp).all()
        return {"ids": ids, "classes": self.readouts["classes"].copy(), "plain_logp": logp}

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: owned_path.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('owned_path.py', r'''
"""Minimal byte-preserved scientific functions extracted from the reviewed owned full-fit path."""


def load_owned_base(base_dir, loading, deadline):
    import torch
    from transformers import AutoModel
    def load_base():
        deadline()
        base, info = AutoModel.from_pretrained(base_dir, dtype=torch.float16, device_map={"": 0},
            attn_implementation="sdpa", trust_remote_code=False, local_files_only=True, output_loading_info=True)
        assert not info.get("missing_keys") and not info.get("mismatched_keys") and not info.get("error_msgs"), info
        assert set(info.get("unexpected_keys", [])) <= {"lm_head.weight"}, info
        assert len(base.layers) == 36 and not hasattr(base, "lm_head")
        assert all(parameter.device.type == "cuda" and parameter.device.index == 0 for parameter in base.parameters())
        base.requires_grad_(False)
        loading.append(info)
        return base
    return load_base()


def make_representation(model, tokenizer, tokens, layers, norms):
    import torch
    def representation(rows):
        batch = tokenizer.pad({"input_ids": [tokens[i] for i in rows]}, return_tensors="pt").to("cuda:0")
        with torch.autocast("cuda", dtype=torch.float16):
            hidden = model(**batch, output_hidden_states=True, use_cache=False).hidden_states
        assert len(hidden) == 37
        pieces = []
        with torch.autocast("cuda", enabled=False):
            for layer, norm in zip(layers, norms):
                mask = batch.attention_mask.unsqueeze(-1).float()
                mask[:, 0] = 0
                assert bool((mask.sum(1) > 0).all())
                mean = (hidden[layer].float() * mask).sum(1) / mask.sum(1)
                pieces.append(norm(mean))
            z = torch.cat(pieces, -1)
        assert bool(torch.isfinite(z).all())
        return z, int(batch.attention_mask.sum()), int(batch.input_ids.numel())
    return representation


def tokenize_owned(base_dir, texts, tokenizer=None):
    if tokenizer is None:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(base_dir, local_files_only=True, trust_remote_code=False)
    assert tokenizer.bos_token_id is None and tokenizer.eos_token_id == 128001
    tokenizer.padding_side = "right"
    tokenizer.pad_token_id = 128004
    tokens = tokenizer(texts, add_special_tokens=False, truncation=True, max_length=447)["input_ids"]
    tokens = [row + [128001] for row in tokens]
    assert all(2 <= len(row) <= 448 for row in tokens)
    return tokenizer, tokens
'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: run_component.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('run_component.py', r'''
"""Remote-only plain-component adapter; no submission, training, or automatic admission."""
import argparse
import json
from pathlib import Path
from component import SmolFinalPlain, validate_contract

parser = argparse.ArgumentParser()
parser.add_argument("--spec", default="inference-spec.json")
parser.add_argument("--artifact-root", required=True)
parser.add_argument("--base-root", required=True)
parser.add_argument("--anchor-train-csv", required=True)
parser.add_argument("--test-csv")
parser.add_argument("--output", default="/kaggle/working/smollm3_plain_logp.npz")
parser.add_argument("--validate-only", action="store_true")
args = parser.parse_args()
spec = json.loads(Path(args.spec).read_text())
validate_contract(spec, args.artifact_root, args.base_root)
if args.validate_only:
    print(json.dumps({"status": "PASS_FULL_FINAL_FILES_NO_MODEL_INIT", "Torch_imported": False}), flush=True)
else:
    assert args.test_csv
    model = SmolFinalPlain(args.spec, args.artifact_root, args.base_root, args.anchor_train_csv)
    import numpy as np
    import pandas as pd
    test = pd.read_csv(args.test_csv, usecols=["id", "text"], dtype=str)
    assert test.notna().all().all() and test.id.is_unique
    result = model.predict_plain_logp(test.id.to_numpy(dtype=str), test.text.tolist())
    np.testing.assert_array_equal(result["ids"], test.id.to_numpy(dtype=str))
    np.savez_compressed(args.output, **result)
    Path(args.output).with_suffix(".json").write_text(json.dumps({"status": "PLAIN_COMPONENT_ONLY",
        "rows": len(test), "anchor_replay": model.anchor_replay,
        "training_or_submission_performed": False, "centering_or_calibration_applied": False}, indent=2))

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: center_readout.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('center_readout.py', r'''
import numpy as np

def source_logsoftmax(value):
    value = value - value.max(1, keepdims=True)
    return value - np.log(np.exp(value).sum(1, keepdims=True))


def add_logprob(p):
    assert np.isfinite(p).all() and (p >= 0).all()
    np.testing.assert_allclose(p.sum(-1), 1, rtol=0, atol=2e-5)
    return np.log(np.clip(p, 1e-9, 1))


def lora_readouts(Z, ro, pools, recovered, adaptive):
    """Production FP32 affine and pool-local query centering, never global centering."""
    assert Z.dtype == np.float32 and Z.shape == (len(Z), 12288)
    assert ro['plain_coef'].dtype == ro['center_coef'].dtype == np.float32
    lg = Z @ ro['plain_coef'].T + ro['plain_intercept']
    lg -= lg.max(1, keepdims=True)
    p = np.exp(lg)
    plain = add_logprob(p / p.sum(1, keepdims=True))
    centered = np.zeros((len(pools), len(Z), 18), dtype=np.float32)
    for context in range(len(pools)):
        for pool in np.unique(pools[context]):
            local = np.flatnonzero((pools[context] == pool) & adaptive[context])
            if not len(local):
                continue
            X = np.asarray(Z[local], dtype=np.float32).copy()
            groups = recovered[context, local]
            for group in np.unique(groups):
                i = groups == group
                X[i] -= X[i].mean(0)
            lc = X @ ro['center_coef'].T + ro['center_intercept']
            centered[context, local] = source_logsoftmax(lc)
    LC = None
    if 'claude_coef' in ro:
        LC = source_logsoftmax(Z @ ro['claude_coef'].T + ro['claude_intercept'])
    return plain, centered, LC



'''[1:-1], crlf=(0, 1))


In [ ]:
'''Name That LLM | Upstream Module: source_balance.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('source_balance.py', r'''
import numpy as np
K = 18
BALANCE_ALPHA = .5

def source_balance(LOGIT):
    P = np.exp(LOGIT - LOGIT.max(1, keepdims=True))
    P /= P.sum(1, keepdims=True)
    if BALANCE_ALPHA > 0 and len(P) >= 2:
        target = (1 - BALANCE_ALPHA) * P.sum(0) + BALANCE_ALPHA * np.full(K, len(P) / K)
        for _ in range(50):
            P *= (target / (P.sum(0) + 1e-12))[None, :]
            P /= P.sum(1, keepdims=True) + 1e-12
        print('balancing changed', int((P.argmax(1) != LOGIT.argmax(1)).sum()), 'of', len(P), flush=True)
    return P

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: compute_center.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('compute_center.py', r'''
"""Optional owned final Smol feature pass and exact pool-local CENTER readout."""
import hashlib
import json
import os
from pathlib import Path
import time

here = Path(__file__).resolve().parent
from admission_preflight import validate
config = validate(here)
assert config['candidate_admitted'] and config['selected_by_root']
assert config['assets_public_audited'] and config['diagnostic_actual_PASS_bound']
assert config['raw_extension'] == 'smol_center_add' and config['coefficient'] == .03
assert os.name != 'nt' and Path('/kaggle/input').is_dir()


def sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


for name, digest in config['source_sha256'].items():
    assert sha(here / name) == digest, name
cache = Path(os.environ['NTL_TABULAR_TEMP']).resolve()
assert cache.is_relative_to(Path('/kaggle/temp').resolve())
assert not (cache / 'smol_center_add.npz').exists(), 'No automatic reuse/retry'
spec = json.loads((here / 'smol-spec.json').read_text())
assert spec['inference_authorized'] and not spec['diagnostic_only_authorized']
assert spec['actual_final_binding']['paired_gain_admitted_by_root']
assets = [p for p in Path('/kaggle/input').rglob('asset_manifest.json')
          if sha(p) == config['compact_asset_manifest_sha256']]
stages = [p for p in Path('/kaggle/input').rglob('smollm3_staging_complete.json')
          if sha(p) == spec['base_manifest_sha256']]
trains = [p for p in Path('/kaggle/input').rglob('train.csv') if sha(p) == spec['train_sha256']]
tests = list(Path('/kaggle/input').rglob('test.csv'))
assert len(assets) == len(stages) == len(trains) == len(tests) == 1

import numpy as np
import pandas as pd
from component import SmolFinalPlain
from center_readout import lora_readouts

with np.load(cache / 'baseline_observation.npz', allow_pickle=False) as values:
    observed = {key: values[key].copy() for key in values.files}
query = pd.read_csv(tests[0], usecols=['id', 'text'], dtype=str)
np.testing.assert_array_equal(query.id.to_numpy(dtype=str), observed['ids'])
assert query.id.is_unique and len(query) > 0
model = SmolFinalPlain(here / 'smol-spec.json', assets[0].parent, stages[0].parent, trains[0])
np.testing.assert_array_equal(model.readouts['classes'], observed['labels'])
import torch
assert torch.cuda.device_count() == 1 and os.environ['CUDA_VISIBLE_DEVICES'].startswith('GPU-')
query_started = time.monotonic()
begin, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
begin.record()
# One query forward pass; _features restores the actual CSV order after stable length sorting.
Z = model._features(query.text.tolist())
end.record()
torch.cuda.synchronize()
query_cuda_ms = float(begin.elapsed_time(end))
assert query_cuda_ms > 0
assert Z.dtype == np.float32 and Z.shape == (len(query), 12288)
pools = observed['pool_ids'][None]
groups = observed['recovered_group_ids'][None]
adaptive = observed['adaptive_mask'][None]
assert pools.shape == groups.shape == adaptive.shape == (1, len(query))
np.testing.assert_array_equal(pools, np.zeros_like(pools))
# The c66c5 body centers separately within each actual recovered adaptive group/pool.
# Its unused plain return is never added to the ensemble and triggers no extra model pass.
_, centered, _ = lora_readouts(Z, model.readouts, pools, groups, adaptive)
SC = centered[0]
assert SC.dtype == np.float32 and SC.shape == (len(query), 18) and np.isfinite(SC).all()
np.testing.assert_array_equal(SC[~adaptive[0]], 0)
np.savez_compressed(cache / 'smol_center_add.npz', ids=observed['ids'], labels=observed['labels'],
                    SC=SC, pool_ids=pools[0], recovered_group_ids=groups[0], adaptive_mask=adaptive[0])
Path('/kaggle/working/smol_center_inference_audit.json').write_text(json.dumps(dict(
    status='OWNED_FINAL_CENTER_FEATURE_PASS_COMPLETE', query_feature_passes=1, query_rows=len(query),
    raw_feature_shape=list(Z.shape), raw_feature_dtype=str(Z.dtype), anchor_replay=model.anchor_replay,
    CUDA_VISIBLE_DEVICES=os.environ['CUDA_VISIBLE_DEVICES'], PID=os.getpid(),
    logical_gpu_name=torch.cuda.get_device_name(0), query_cuda_event_ms=query_cuda_ms,
    query_wall_seconds=time.monotonic()-query_started,
    center_helper_sha256=sha(here / 'center_readout.py'), groups_reclustered=False,
    centers_within_actual_pools_and_recovered_groups=True, zero_outside_adaptive=True,
    uses_plain_readout=False, training=False, target_column_read=False), indent=2))

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: finish_center_protected.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('finish_center_protected.py', r'''
"""Optional raw+.03*SC, original alpha.5 balance/gate, then shared protected PFN.10."""
import hashlib
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
from claude_gate import gated_predict, load_gate_state
from protected_operator import protected
from source_balance import source_balance

here = Path(__file__).resolve().parent
from admission_preflight import validate
config = validate(here)
assert config['candidate_admitted'] and config['selected_by_root']
assert config['assets_public_audited'] and config['diagnostic_actual_PASS_bound']
assert config['raw_extension'] == 'smol_center_add' and config['coefficient'] == .03
assert config['alpha'] == .5 and config['protected_weight'] == .10 and config['post'] == 0
for name, digest in config['source_sha256'].items():
    assert hashlib.sha256((here / name).read_bytes()).hexdigest() == digest, name
cache = Path(os.environ['NTL_TABULAR_TEMP']).resolve()
assert cache.is_relative_to(Path('/kaggle/temp').resolve())


def load(name):
    with np.load(cache / name, allow_pickle=False) as values:
        return {key: values[key].copy() for key in values.files}


observed, smol, pfn = (load(name) for name in
    ('baseline_observation.npz', 'smol_center_add.npz', 'tabpfn_proba.npz'))
raw, P, ids, labels = (observed[key] for key in ('raw', 'balanced', 'ids', 'labels'))
assert raw.dtype == P.dtype == np.float64 and raw.shape == P.shape == (len(ids), 18)
for values in (smol, pfn):
    np.testing.assert_array_equal(values['ids'], ids)
    np.testing.assert_array_equal(values['labels'], labels)
for key in ('pool_ids', 'recovered_group_ids', 'adaptive_mask'):
    np.testing.assert_array_equal(smol[key], observed[key])
# The extracted original source balance must reproduce the observed baseline bit-for-bit.
np.testing.assert_array_equal(source_balance(raw), P)
gate = load_gate_state(cache, P, ids, labels)
baseline = gated_predict(P, labels, **gate)
np.testing.assert_array_equal(baseline, observed['gated_predictions'])
control = pd.read_parquet('/kaggle/working/B556_gated_control.parquet')
np.testing.assert_array_equal(control.id.to_numpy(dtype=str), ids)
np.testing.assert_array_equal(control.model.to_numpy(dtype=str), labels[baseline])
SC = smol['SC']
assert SC.dtype == np.float32 and SC.shape == raw.shape and np.isfinite(SC).all()
np.testing.assert_array_equal(SC[~observed['adaptive_mask']], 0)
# Same FP32 multiplication and promotion order as the actual c66c5 conditional CPU arm.
extended_raw = raw + .03 * SC
q = source_balance(extended_raw)
extended_labels = gated_predict(q, labels, **gate)
claude = gate['CLAUDE_IDX']
nonclaude = np.flatnonzero(~np.char.startswith(labels, 'claude'))
assert len(claude) == 5 and len(nonclaude) == 13
tabular = pfn['proba']
assert tabular.dtype == np.float64 and tabular.shape == q.shape and np.isfinite(tabular).all()
candidate, prediction, reverted = protected(q[None], tabular, extended_labels[None], claude, nonclaude, weight=.10)
submission = pd.DataFrame(dict(id=ids, model=labels[prediction[0]]))
assert submission.id.is_unique and len(submission) == len(control) and submission.model.isin(labels).all()
submission.to_parquet('/kaggle/working/submission.parquet', index=False)
Path('/kaggle/working/protected_residual_audit.json').write_text(json.dumps(dict(
    status='SMOL_CENTER_ADD_PROTECTED_PFN10_PREDICTIONS_COMPLETE', baseline_balance_and_gate_replay_bitexact=True,
    formula='original_raw + .03*SC -> original_alpha.5_balance -> original_Claude_gate -> protected_PFN.10',
    source_notebook_id=139200100, source_version=2, coefficient=.03, alpha=.5, post=0,
    residual_weight=.10, query_rows=len(ids), extra_global_normalization=False,
    protected_family_membership_relative_to='SmolCENTER extended original gate decisions',
    reverted_family_flips=int(reverted.sum()), training=False), indent=2))

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: admission_preflight.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('admission_preflight.py', r'''
"""Stdlib-only immutable diagnostic evidence and explicit candidate admission guards."""
import hashlib
import json
from pathlib import Path


def validate(here):
    here = Path(here)
    config = json.loads((here / 'optional-config.json').read_text())
    assert config['candidate_admitted'] and config['selected_by_root']
    assert config['assets_public_audited'] and config['diagnostic_actual_PASS_bound']
    assert config['root_decision_binding'] and config['root_decision_binding']['explicit_candidate_admission']
    binding = config['diagnostic_PASS_binding']
    assert binding['notebook_id'] == 139353455 and binding['version'] == 3 and binding['script_version_id'] == 359581988
    raw = (here / 'actual-diagnostic-audit.json').read_bytes()
    assert hashlib.sha256(raw).hexdigest() == binding['audit_sha256']
    audit = json.loads(raw)
    assert audit['notebook_id'] == binding['notebook_id'] and audit['version'] == 3
    assert audit['component_sha256'] == config['source_sha256']['component.py']
    assert audit['both_physical_T4_forward_CUDA_events_proven'] and audit['training_updates'] == 0
    assert not audit['candidate_or_production_admitted']
    for rows in ('3', '5'):
        report = audit['reports'][rows]
        assert report['status'] == 'PASS_INFERENCE_DIAGNOSTIC_ONLY'
        assert report['component_sha256'] == config['source_sha256']['component.py']
        assert report['owned_path_sha256'] == config['source_sha256']['owned_path.py']
        assert report['observed_query_batch_sizes'] == ([3] if rows == '3' else [3, 2])
        assert report['prediction_cuda_event_ms'] > 0 and report['finite'] and report['id_order_preserved']
        replay = report['anchor_replay']
        assert replay['feature_rtol'] == replay['feature_atol'] == 1e-5 and replay['probability_atol'] == 1e-6
        assert replay['feature_maxabs'] == 0 and replay['probability_maxabs'] == {'plain': 0, 'center': 0}
    return config

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: run_parallel.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('run_parallel.py', r'''
"""Optional UUID-isolated GPU0 PFN128 and GPU1 owned Smol; no shared-GPU loading."""
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import time

here = Path(__file__).resolve().parent
optional = json.loads((here / 'optional-config.json').read_text())
config = json.loads((here / 'runtime_config.json').read_text())
assert optional['candidate_admitted'] and optional['selected_by_root'] and config['candidate_admitted']
assert optional['assets_public_audited'] and optional['diagnostic_actual_PASS_bound']
assert config['raw_extension'] == optional['raw_extension'] == 'smol_center_add'
assert config['tabpfn']['n_estimators'] == 128
assert os.name != 'nt' and Path('/kaggle/input').is_dir()
mapping = []
for line in subprocess.check_output(['nvidia-smi', '--query-gpu=index,uuid,name',
                                     '--format=csv,noheader,nounits'], text=True).strip().splitlines():
    index, uuid, name = [value.strip() for value in line.split(',', 2)]
    mapping.append(dict(physical_index=int(index), uuid=uuid, name=name))
mapping.sort(key=lambda gpu: gpu['physical_index'])
assert [gpu['physical_index'] for gpu in mapping] == [0, 1]
assert all('T4' in gpu['name'] and gpu['uuid'].startswith('GPU-') for gpu in mapping)
assert mapping[0]['uuid'] != mapping[1]['uuid']
out = Path('/kaggle/working/smol_pfn_parallel')
out.mkdir(exist_ok=False)
children, logs, records = [], [], []
started = time.monotonic()
try:
    for index, route in enumerate(('PFN128', 'SmolCENTER')):
        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = mapping[index]['uuid']
        env['HF_HUB_OFFLINE'] = env['TRANSFORMERS_OFFLINE'] = '1'
        # Preserve the already installed PFN-only add-on environment; Smol uses the original runtime.
        if route == 'PFN128':
            env['PYTHONPATH'] = os.environ['NTL_TABULAR_SITE']
        command = ([sys.executable, '-u', str(here / 'tabular_worker.py'), 'tabpfn'] if index == 0
                   else [sys.executable, '-u', str(here / 'compute_center.py')])
        log = (out / (route + '.log')).open('w', buffering=1)
        logs.append(log)
        child = subprocess.Popen(command, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        children.append(child)
        records.append(dict(route=route, PID=child.pid, GPU=mapping[index],
                            started_seconds=time.monotonic()-started, finished_seconds=None))
    tick, overlap_samples = 0, 0
    observed_compute_PIDs = set()
    while True:
        elapsed = time.monotonic()-started
        codes = [child.poll() for child in children]
        for i, code in enumerate(codes):
            if code is not None and records[i]['finished_seconds'] is None:
                records[i]['finished_seconds'] = elapsed
            if code is None:
                limit = config['model_timeout_seconds'] if i == 0 else optional['smol_timeout_seconds']
                assert elapsed < limit, (records[i]['route'], 'timeout; no automatic retry')
        if any(code is not None and code != 0 for code in codes):
            raise RuntimeError(f'Parallel worker failed{codes}; no finalization or automatic retry')
        sample = subprocess.check_output(['nvidia-smi', '--query-gpu=index,uuid,utilization.gpu,memory.used',
                                         '--format=csv,noheader,nounits'], text=True, timeout=15)
        apps = subprocess.check_output(['nvidia-smi', '--query-compute-apps=pid,gpu_uuid,used_memory',
                                       '--format=csv,noheader,nounits'], text=True, timeout=15)
        active = {}
        for line in sample.strip().splitlines():
            index, uuid, utilization, memory = [value.strip() for value in line.split(',')]
            active[uuid] = float(utilization) > 0
        matched = []
        for record in records:
            exact = any(fields[0].strip() == str(record['PID']) and fields[1].strip() == record['GPU']['uuid']
                        for fields in (line.split(',') for line in apps.strip().splitlines()) if len(fields) >= 2)
            if exact:
                observed_compute_PIDs.add(record['PID'])
            matched.append(exact)
        both_compute = all(code is None for code in codes) and all(matched) and all(active.get(gpu['uuid'], False) for gpu in mapping)
        overlap_samples += int(both_compute)
        with (out / 'gpu-telemetry.jsonl').open('a') as handle:
            handle.write(json.dumps(dict(elapsed_seconds=elapsed, GPU=sample, compute_apps=apps,
                                         worker_codes=codes, both_workers_matched_active=both_compute))+'\n')
        if tick % 6 == 0:
            print(json.dumps(dict(stage='PFN_SMOL_PROGRESS', elapsed_seconds=elapsed, workers=records,
                                  return_codes=codes, devices=sample)), flush=True)
        if all(code == 0 for code in codes):
            break
        tick += 1
        time.sleep(5)
    assert all(record['PID'] in observed_compute_PIDs for record in records), 'Actual per-worker GPU mapping not observed'
    report = json.loads(Path('/kaggle/working/smol_center_inference_audit.json').read_text())
    assert report['status'] == 'OWNED_FINAL_CENTER_FEATURE_PASS_COMPLETE' and report['query_cuda_event_ms'] > 0
    assert report['CUDA_VISIBLE_DEVICES'] == mapping[1]['uuid'] and report['PID'] == children[1].pid
    (out / 'COMPLETE.json').write_text(json.dumps(dict(status='BOTH_OPTIONAL_COMPONENTS_COMPLETE', workers=records,
        overlap_active_samples=overlap_samples, actual_compute_overlap_observed=overlap_samples > 0,
        missing_sample_caveat=None if overlap_samples else 'No overlapping active sample captured; no claim of measured compute overlap.',
        wall_seconds=time.monotonic()-started, automatic_retry=False), indent=2))
finally:
    for child in children:
        if child.poll() is None:
            os.killpg(child.pid, signal.SIGTERM)
    for child in children:
        try:
            child.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(child.pid, signal.SIGKILL)
            child.wait()
    for log in logs:
        log.close()

'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: run_candidate_optional.py
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('run_candidate_optional.py', r'''
"""Offline sequential baseline, fixed tabular members and CPU finalization."""
import hashlib
import json
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

assert os.name!='nt' and Path('/kaggle/input').is_dir(),'Kaggle execution only'
here=Path(__file__).resolve().parent
config=json.loads((here/'runtime_config.json').read_text())
assert config['reference_scored_script_version_id']==359313348
assert config['candidate_admitted'], 'Candidate admission required'
assert config['protected_weight']==.10 and config['raw_extension'] == 'smol_center_add'
assert not config['second_column_balance'] and config['probability_transform']=='none'
matches=list(Path('/kaggle/input').rglob(config['tabpfn']['checkpoint_filename']))
assert len(matches)==1,'Attach the exact reviewed offline tabular asset package'
asset=matches[0].parent
manifest=json.loads((asset/'runtime_manifest.json').read_text())
assert manifest['status']=='CPU_SETUP_COMPLETE_NO_MODEL_EXECUTION'
assert manifest['global_environment_unchanged'] and not manifest['model_fit'] and not manifest['model_predict']
assert manifest['checkpoint_sha256']==config['tabpfn']['checkpoint_sha256']
assert matches[0].stat().st_size==config['tabpfn']['checkpoint_bytes']
with matches[0].open('rb') as stream:
    assert hashlib.file_digest(stream,'sha256').hexdigest()==config['tabpfn']['checkpoint_sha256']
lock=asset/'runtime-addon-lock.txt'
wheels=asset
assert lock.is_file() and any(wheels.glob('*.whl')),'Reviewed offline addon lock/wheels are required'
for item in manifest['selected_addons'].values():
    with (wheels/item['wheel']).open('rb') as stream:
        assert hashlib.file_digest(stream,'sha256').hexdigest()==item['sha256']
for line in lock.read_text().splitlines():
    if '==' in line and not line.lstrip().startswith('#'):
        package=re.split(r'==',line.strip(),maxsplit=1)[0].lower().replace('_','-')
        assert package not in ('torch','transformers','peft','accelerate','safetensors')
        assert not package.startswith('nvidia-'),'Reuse the original CUDA/Torch runtime'
Path('/kaggle/temp').mkdir(parents=True,exist_ok=True)
temp=Path(tempfile.mkdtemp(prefix='ntl-tabular-',dir='/kaggle/temp'))
site=temp/'site'
base_env=os.environ.copy()
base_env['NTL_TABULAR_TEMP']=str(temp)
base_env['NTL_TABULAR_ASSET']=str(asset)
subprocess.run([sys.executable,'-m','pip','install','--no-index','--find-links',str(wheels),
    '--no-deps','--require-hashes','--target',str(site),'-r',str(lock)],
    check=True,timeout=config['package_install_timeout_seconds'])
tabular_env=base_env.copy()
tabular_env['PYTHONPATH']=str(site)
tabular_env['HF_HUB_OFFLINE']='1'
subprocess.run([sys.executable,'-c',
    "import json, importlib.metadata as m; from pathlib import Path; import numpy, pandas, scipy, sklearn, torch; from tabpfn import TabPFNClassifier; from tabpfn.constants import ModelVersion; actual=dict(numpy=numpy.__version__, pandas=pandas.__version__, scipy=scipy.__version__, sklearn=sklearn.__version__, torch=torch.__version__, tabpfn=m.version('tabpfn')); expected="+repr(config['verified_runtime_imports'])+"; assert actual==expected, (actual,expected); assert ModelVersion.V3_5; Path('/kaggle/working/tabular_environment_audit.json').write_text(json.dumps(dict(status='VERIFIED_OFFLINE_RUNTIME_IMPORTS',imports=actual,diagnostic_script_version_id=359162368),indent=2)); print('Offline TabPFN imports passed', flush=True)"],
    env=tabular_env,check=True,timeout=90)
subprocess.run([sys.executable,str(here/'run_baseline.py')],cwd='/kaggle/working',env=base_env,
    check=True,timeout=config['baseline_timeout_seconds'])
baseline=Path('/kaggle/working/submission.parquet')
assert baseline.is_file()
control=Path('/kaggle/working/B556_gated_control.parquet')
assert not control.exists()
baseline.rename(control)
subprocess.run([sys.executable,str(here/'tabular_worker.py'),'prepare'],env=tabular_env,
    check=True,timeout=config['prepare_timeout_seconds'])
base_env['NTL_TABULAR_SITE']=str(site)
subprocess.run([sys.executable,'-u',str(here/'run_parallel.py')],env=base_env,check=True,
    timeout=max(config['model_timeout_seconds'],3600)+120)
subprocess.run([sys.executable,str(here/'finish_center_protected.py')],env=base_env,check=True,timeout=120)
assert Path('/kaggle/working/submission.parquet').is_file()
assert json.loads(Path('/kaggle/working/protected_residual_audit.json').read_text())['status']=='SMOL_CENTER_ADD_PROTECTED_PFN10_PREDICTIONS_COMPLETE'
'''[1:-1], crlf=())


In [ ]:
'''Name That LLM | Upstream Module: actual-diagnostic-audit.json
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('actual-diagnostic-audit.json', r'''
{
  "notebook_sha256": "8b95573e0fc61e61ee4ae96f443112ccff9a2c10df4929ec669d0e9b3c98b77d",
  "telemetry_samples": 48,
  "targets_loaded": false,
  "compact_manifest_sha256": "1772703faaee5d2b30e72fe37d0802c0c20eab2332e4d67a061cda6912fb611a",
  "local_model_execution": false,
  "optimizer_loaded": false,
  "notebook_id": 139353455,
  "version": 3,
  "ref": "lightningv08/private-smollm3-final-inference-diagnostic",
  "training_updates": 0,
  "component_sha256": "ab1df94cea1358d95125ed99f4fbe9606f7e0690ca6b6938ead419f884343f46",
  "candidate_or_production_admitted": false,
  "quality_score_computed": false,
  "reports": {
    "5": {
      "status": "PASS_INFERENCE_DIAGNOSTIC_ONLY",
      "rows": 5,
      "classes": [
        "claude-fable-5",
        "claude-opus-4-6",
        "claude-opus-4-8",
        "claude-opus-5",
        "claude-sonnet-5",
        "gemini-3.1-pro-preview",
        "gemini-3.6-flash",
        "gemini-3.7-flash",
        "gemini-3.8-flash",
        "glm-5.3",
        "glm-5.3-flash",
        "gpt-5.5",
        "gpt-5.6-luna",
        "gpt-5.6-sol",
        "gpt-5.6-terra",
        "gpt-6-astra",
        "kimi-k3",
        "qwen3.8-max"
      ],
      "ids": [
        "0018b3765972",
        "0041ac86ee93",
        "005a19083eae",
        "009319596f10",
        "00d3e8c92596"
      ],
      "output_shape": [
        5,
        18
      ],
      "dtype": "float32",
      "finite": true,
      "id_order_preserved": true,
      "observed_query_batch_sizes": [
        3,
        2
      ],
      "canonical_anchor_batch_size": 3,
      "anchor_replay": {
        "feature_maxabs": 0.0,
        "probability_maxabs": {
          "plain": 0.0,
          "center": 0.0
        },
        "feature_rtol": 1E-05,
        "feature_atol": 1E-05,
        "probability_atol": 1E-06
      },
      "CUDA_VISIBLE_DEVICES": "GPU-ae8fbdfe-6b4a-364c-0902-1ce889cf5b2e",
      "logical_gpu_uuid": "ae8fbdfe-6b4a-364c-0902-1ce889cf5b2e",
      "logical_gpu_name": "Tesla T4",
      "prediction_cuda_event_ms": 403.59222412109375,
      "load_and_anchor_wall_seconds": 90.81032806999997,
      "peak_allocated_bytes": 6454882304,
      "peak_reserved_bytes": 6603931648,
      "full_checkpoint_binding": {
        "actual_full_QA_passed": true,
        "paired_gain_admitted_by_root": false,
        "notebook_id": 139347470,
        "version": 1,
        "script_version_id": 359564738,
        "worker_sha256": "4dea00c2313dad426ba1081f3b571540c84f6e5e441c57075699e6f9557e935b",
        "checkpoint_relative": "final/finished_checkpoint",
        "complete_sha256": "31b52250f5eaab7cbd9b60152e4471455566b6d40fae911f0f1025ac827854fb",
        "report_sha256": "53be4d9394bd161ed0d58b75ecdc39f8dafb40e12dec8d9ad5b3e76d42caf784",
        "portable_files_sha256": {
          "adapter/adapter_model.safetensors": "acc77f2e95d86e0edc541afa46c1af16ae957bed955435c5319a4dc8a60b4c41",
          "adapter/adapter_config.json": "fe64dfa8471149025ec910d7a673c03ee673827041efc1cda0988dec16b42af3",
          "heads.safetensors": "cbc50ca2abc66a6002459a2343b02c0598e8d94fe0954303959d297ae2c2fe8b",
          "readouts.npz": "04115910b24569c5e08641342b0cefe93c8156515f93d0b920ede01dfba2ddc2",
          "replay_anchors.npz": "e49a747a32a5529f2d0e1363a3f5b7d9e1ca34d7fc114571ec624277207a29b2",
          "adapter/README.md": "436049a05f1f088661ddcdc33c42b1e139d5ed147de162b74b178c73db689bfb"
        },
        "full_qa_audit_sha256": "9742ec63fd13211f000c8e464c3e88e197eb032f8a085cb57b40e5d4fc154a7e"
      },
      "component_sha256": "ab1df94cea1358d95125ed99f4fbe9606f7e0690ca6b6938ead419f884343f46",
      "owned_path_sha256": "20dabafd3f423196863ce3d1fbd4caa8b196bd517a88422561615db003d1e620",
      "worker_sha256": "6d3e3ffe41c39d731412e668f94afdba411eacee31425530ac22966d86429fb7",
      "output_sha256": "5df6ec951397e1fca210f4b154f3342a7577671e369b60c84ffc6ad858c861e1",
      "training_updates": 0,
      "targets_loaded": false,
      "optimizer_loaded": false,
      "quantization": false,
      "candidate_or_production_admitted": false,
      "wall_seconds": 91.395060592
    },
    "3": {
      "status": "PASS_INFERENCE_DIAGNOSTIC_ONLY",
      "rows": 3,
      "classes": [
        "claude-fable-5",
        "claude-opus-4-6",
        "claude-opus-4-8",
        "claude-opus-5",
        "claude-sonnet-5",
        "gemini-3.1-pro-preview",
        "gemini-3.6-flash",
        "gemini-3.7-flash",
        "gemini-3.8-flash",
        "glm-5.3",
        "glm-5.3-flash",
        "gpt-5.5",
        "gpt-5.6-luna",
        "gpt-5.6-sol",
        "gpt-5.6-terra",
        "gpt-6-astra",
        "kimi-k3",
        "qwen3.8-max"
      ],
      "ids": [
        "0018b3765972",
        "0041ac86ee93",
        "005a19083eae"
      ],
      "output_shape": [
        3,
        18
      ],
      "dtype": "float32",
      "finite": true,
      "id_order_preserved": true,
      "observed_query_batch_sizes": [
        3
      ],
      "canonical_anchor_batch_size": 3,
      "anchor_replay": {
        "feature_maxabs": 0.0,
        "probability_maxabs": {
          "plain": 0.0,
          "center": 0.0
        },
        "feature_rtol": 1E-05,
        "feature_atol": 1E-05,
        "probability_atol": 1E-06
      },
      "CUDA_VISIBLE_DEVICES": "GPU-0dbd0856-9926-3259-e839-1869a6545846",
      "logical_gpu_uuid": "0dbd0856-9926-3259-e839-1869a6545846",
      "logical_gpu_name": "Tesla T4",
      "prediction_cuda_event_ms": 258.1510925292969,
      "load_and_anchor_wall_seconds": 90.81061158,
      "peak_allocated_bytes": 6454882304,
      "peak_reserved_bytes": 6603931648,
      "full_checkpoint_binding": {
        "actual_full_QA_passed": true,
        "paired_gain_admitted_by_root": false,
        "notebook_id": 139347470,
        "version": 1,
        "script_version_id": 359564738,
        "worker_sha256": "4dea00c2313dad426ba1081f3b571540c84f6e5e441c57075699e6f9557e935b",
        "checkpoint_relative": "final/finished_checkpoint",
        "complete_sha256": "31b52250f5eaab7cbd9b60152e4471455566b6d40fae911f0f1025ac827854fb",
        "report_sha256": "53be4d9394bd161ed0d58b75ecdc39f8dafb40e12dec8d9ad5b3e76d42caf784",
        "portable_files_sha256": {
          "adapter/adapter_model.safetensors": "acc77f2e95d86e0edc541afa46c1af16ae957bed955435c5319a4dc8a60b4c41",
          "adapter/adapter_config.json": "fe64dfa8471149025ec910d7a673c03ee673827041efc1cda0988dec16b42af3",
          "heads.safetensors": "cbc50ca2abc66a6002459a2343b02c0598e8d94fe0954303959d297ae2c2fe8b",
          "readouts.npz": "04115910b24569c5e08641342b0cefe93c8156515f93d0b920ede01dfba2ddc2",
          "replay_anchors.npz": "e49a747a32a5529f2d0e1363a3f5b7d9e1ca34d7fc114571ec624277207a29b2",
          "adapter/README.md": "436049a05f1f088661ddcdc33c42b1e139d5ed147de162b74b178c73db689bfb"
        },
        "full_qa_audit_sha256": "9742ec63fd13211f000c8e464c3e88e197eb032f8a085cb57b40e5d4fc154a7e"
      },
      "component_sha256": "ab1df94cea1358d95125ed99f4fbe9606f7e0690ca6b6938ead419f884343f46",
      "owned_path_sha256": "20dabafd3f423196863ce3d1fbd4caa8b196bd517a88422561615db003d1e620",
      "worker_sha256": "6d3e3ffe41c39d731412e668f94afdba411eacee31425530ac22966d86429fb7",
      "output_sha256": "a73cc77ac0e8be5b6916332ca363047c0b453f22120ba8ff34e893190508504a",
      "training_updates": 0,
      "targets_loaded": false,
      "optimizer_loaded": false,
      "quantization": false,
      "candidate_or_production_admitted": false,
      "wall_seconds": 91.24247906099998
    }
  },
  "status": "PASS_ACTUAL_SMOL_FINAL_INFERENCE_BOTH_T4_STRICT_ORIGINAL_ANCHORS",
  "overlap_samples": 4,
  "script_version_id": 359581988,
  "root": "D:\\Programming\\Python AI ML\\Kaggle\\name-that-llm\\experiments\\smollm3-lora\\inference\\smoke\\results-v3",
  "actualsource_bytechecks": [
    {
      "exact": true,
      "sha256": "ab1df94cea1358d95125ed99f4fbe9606f7e0690ca6b6938ead419f884343f46",
      "source": "component.py"
    },
    {
      "exact": true,
      "sha256": "20dabafd3f423196863ce3d1fbd4caa8b196bd517a88422561615db003d1e620",
      "source": "owned_path.py"
    },
    {
      "exact": true,
      "sha256": "8d6ee8631ffa99baaf3f9fdb49e3d8a4dbd6dbe2284e2ea1617ea5965902052b",
      "source": "run_component.py"
    },
    {
      "exact": true,
      "sha256": "6d3e3ffe41c39d731412e668f94afdba411eacee31425530ac22966d86429fb7",
      "source": "worker.py"
    },
    {
      "exact": true,
      "sha256": "8f662045e5713119c929f89fa87d07fd397c5818f9f4b98d1ea1136ed6b3fe50",
      "source": "run_diagnostic.py"
    },
    {
      "exact": true,
      "sha256": "8fd38892579434daba9bb0009113766338f891bae2af274de835493309ffbbe4",
      "source": "diagnostic-spec.json"
    }
  ],
  "headers": {
    "5": {
      "ids.npy": "{'descr': '<U12', 'fortran_order': False, 'shape': (5,), }",
      "plain_logp.npy": "{'descr': '<f4', 'fortran_order': False, 'shape': (5, 18), }",
      "classes.npy": "{'descr': '<U22', 'fortran_order': False, 'shape': (18,), }"
    },
    "3": {
      "ids.npy": "{'descr': '<U12', 'fortran_order': False, 'shape': (3,), }",
      "plain_logp.npy": "{'descr': '<f4', 'fortran_order': False, 'shape': (3, 18), }",
      "classes.npy": "{'descr': '<U22', 'fortran_order': False, 'shape': (18,), }"
    }
  },
  "checked_utc": "2026-10-18T04:58:37.5249658Z",
  "strict_numerical_anchor_replay_passed": true,
  "parent_complete": {
    "status": "PASS_BOTH_INFERENCE_FIXTURES_ONLY",
    "rows": [
      3,
      5
    ],
    "GPU_mapping": [
      {
        "physical_index": 0,
        "uuid": "GPU-0dbd0856-9926-3259-e839-1869a6545846",
        "name": "Tesla T4"
      },
      {
        "physical_index": 1,
        "uuid": "GPU-ae8fbdfe-6b4a-364c-0902-1ce889cf5b2e",
        "name": "Tesla T4"
      }
    ],
    "candidate_or_production_admitted": false,
    "training_updates": 0,
    "wall_seconds": 96.44504450000005
  },
  "actual_compact_dataset": "lightningv08/ntl-smollm3-final-inference-assets/1",
  "both_physical_T4_forward_CUDA_events_proven": true,
  "telemetry_derived_from_actual_nvidia_smi_csv": true,
  "max_sampled_utilization_percent": {
    "1": 9.0,
    "0": 8.0
  },
  "max_sampled_memory_mib": {
    "1": 6431.0,
    "0": 6431.0
  },
  "CUDA_event_proof_not_dependent_on_2sec_sampler": true
}

'''[1:-1], crlf=True)


In [ ]:
'''Name That LLM | Upstream Module: selection-evidence.json
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('selection-evidence.json', r'''
{
  "selected_by_root": true,
  "selected_branch": "smol_center_add_protected_pfn10",
  "notebook_id": 139355129,
  "version": 2,
  "script_version_id": 359583421,
  "report_sha256": "f26285d0e463ed814de463cc190574d7d983923bb4087a38e423866e8ec73676",
  "actual_arm": {
    "arm": "Smol_center_add_PFN10",
    "context_BA": [
      0.5210727969348659,
      0.5505108556832694,
      0.5988505747126436
    ],
    "mean_BA": 0.5568114091102597,
    "comparisons": {
      "B556": {
        "control_context_BA": [
          0.5118135376756067,
          0.5411877394636014,
          0.596934865900383
        ],
        "control_mean_BA": 0.5499787143465306,
        "context_delta": [
          0.009259259259259257,
          0.009323116219667945,
          0.0019157088122605341
        ],
        "mean_delta": 0.006832694763729246,
        "classwise_context_recall_delta": [
          [
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            -0.033333333333333326,
            0.0,
            0.0,
            0.06666666666666665,
            0.033333333333333326,
            0.033333333333333326,
            0.0,
            0.033333333333333326,
            0.0,
            0.033333333333333326
          ],
          [
            0.03448275862068964,
            0.0,
            0.0,
            0.0,
            0.0,
            0.033333333333333326,
            0.06666666666666665,
            0.03333333333333338,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0
          ],
          [
            0.03448275862068961,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0
          ]
        ],
        "classwise_mean_recall_delta": [
          0.02298850574712642,
          0.0,
          0.0,
          0.0,
          0.0,
          0.011111111111111108,
          0.022222222222222216,
          0.011111111111111127,
          -0.011111111111111108,
          0.0,
          0.0,
          0.022222222222222216,
          0.011111111111111108,
          0.011111111111111108,
          0.0,
          0.011111111111111108,
          0.0,
          0.011111111111111108
        ],
        "corrected_per_context": [
          7,
          5,
          2
        ],
        "broken_per_context": [
          2,
          0,
          1
        ]
      },
      "protected_PFN_only": {
        "control_context_BA": [
          0.5210727969348659,
          0.546743295019157,
          0.5969348659003831
        ],
        "control_mean_BA": 0.5549169859514687,
        "context_delta": [
          0.0,
          0.00376756066411239,
          0.0019157088122605278
        ],
        "mean_delta": 0.0018944231587909727,
        "classwise_context_recall_delta": [
          [
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0
          ],
          [
            0.03448275862068964,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.033333333333333326,
            0.03333333333333338,
            0.0,
            0.0,
            0.0,
            0.0,
            -0.033333333333333326,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0
          ],
          [
            0.03448275862068961,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.033333333333333326,
            0.0,
            0.0,
            0.0,
            -0.03333333333333344,
            0.0
          ]
        ],
        "classwise_mean_recall_delta": [
          0.02298850574712642,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.011111111111111108,
          0.011111111111111127,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          -0.011111111111111146,
          0.0
        ],
        "corrected_per_context": [
          0,
          3,
          2
        ],
        "broken_per_context": [
          0,
          1,
          1
        ]
      },
      "Gemma_only": {
        "control_context_BA": [
          0.5136653895274583,
          0.5412515964240101,
          0.5988505747126436
        ],
        "control_mean_BA": 0.5512558535547041,
        "context_delta": [
          0.007407407407407406,
          0.009259259259259257,
          0.0
        ],
        "mean_delta": 0.005555555555555554,
        "classwise_context_recall_delta": [
          [
            0.0,
            -0.033333333333333326,
            0.0,
            0.0,
            0.0,
            0.0,
            -0.033333333333333326,
            0.0,
            0.0,
            0.0,
            0.033333333333333326,
            0.033333333333333326,
            0.033333333333333326,
            0.033333333333333326,
            0.0,
            0.033333333333333326,
            0.0,
            0.033333333333333326
          ],
          [
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.033333333333333326,
            0.06666666666666665,
            0.0,
            0.0,
            0.0,
            0.033333333333333326,
            0.033333333333333326,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0
          ],
          [
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            -0.033333333333333326,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.033333333333333326,
            0.0
          ]
        ],
        "classwise_mean_recall_delta": [
          0.0,
          -0.011111111111111108,
          0.0,
          0.0,
          0.0,
          0.011111111111111108,
          0.011111111111111108,
          0.0,
          0.0,
          -0.011111111111111108,
          0.022222222222222216,
          0.022222222222222216,
          0.011111111111111108,
          0.011111111111111108,
          0.0,
          0.011111111111111108,
          0.011111111111111108,
          0.011111111111111108
        ],
        "corrected_per_context": [
          7,
          6,
          2
        ],
        "broken_per_context": [
          3,
          1,
          2
        ]
      },
      "Smol_center_only": {
        "control_context_BA": [
          0.5118135376756067,
          0.5468071519795656,
          0.6007024265644954
        ],
        "control_mean_BA": 0.553107705406556,
        "context_delta": [
          0.009259259259259257,
          0.003703703703703703,
          -0.0018518518518518515
        ],
        "mean_delta": 0.003703703703703703,
        "classwise_context_recall_delta": [
          [
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            -0.033333333333333326,
            0.0,
            0.0,
            0.06666666666666665,
            0.033333333333333326,
            0.033333333333333326,
            0.0,
            0.033333333333333326,
            0.0,
            0.033333333333333326
          ],
          [
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.033333333333333326,
            0.06666666666666665,
            0.0,
            -0.033333333333333326,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0
          ],
          [
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            -0.033333333333333326,
            0.0,
            0.0,
            -0.033333333333333326,
            0.0,
            0.0,
            0.0,
            0.033333333333333326,
            0.0
          ]
        ],
        "classwise_mean_recall_delta": [
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.011111111111111108,
          0.022222222222222216,
          0.0,
          -0.022222222222222216,
          -0.011111111111111108,
          0.0,
          0.022222222222222216,
          0.0,
          0.011111111111111108,
          0.0,
          0.011111111111111108,
          0.011111111111111108,
          0.011111111111111108
        ],
        "corrected_per_context": [
          7,
          3,
          1
        ],
        "broken_per_context": [
          2,
          1,
          2
        ]
      }
    },
    "family_reverts_per_context": [
      2,
      2,
      0
    ]
  },
  "helper_sha256": "9d9f8a89b92935075131d1e3a26a7948cbbe6eb1495d5ca808329a5ecda3184a",
  "main18_sha256": "82ab07432a63d6767fe3b98e5840dd03c9d3166dc653ef2498d840896de3c60b",
  "candidate_admitted": false,
  "public_assets_audit_passed": false,
  "interpretation": "Root selection from one reused539-row30-prompt conditional readout; not full-fivefold OOF, independent holdout, CI or leaderboard forecast"
}
'''[1:-1], crlf=True)


In [ ]:
'''Name That LLM | Upstream Module: runtime_config.json
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('runtime_config.json', r'''
{
  "reference_notebook": "obstacledeveloper/local-candidate-adaptive-centered-stylegram",
  "reference_version": 2,
  "reference_scored_script_version_id": 359313348,
  "reference_source_sha256": "b856e736d5cb47d8fa309ce2acd5ce0588a4f91478bdeae99c491f31b7c26a4c",
  "tabpfn": {
    "checkpoint_filename": "tabpfn-v3.5-20260909.safetensors",
    "checkpoint_sha256": "ece4d67eadfea42eb0e610df5189bea60cb7f31073d81e9c7a019b76eacf0be3",
    "checkpoint_bytes": 876027932,
    "package_version": "9.0.0",
    "n_estimators": 128
  },
  "offline_asset_directory_name": "ntl-tabular-offline-assets",
  "pca_components_per_view": 64,
  "pca_iterated_power": 3,
  "random_state": 42,
  "probability_transform": "none",
  "orthography_post_penalty": 0,
  "second_column_balance": false,
  "baseline_timeout_seconds": 13200,
  "prepare_timeout_seconds": 300,
  "model_timeout_seconds": 3600,
  "package_install_timeout_seconds": 300,
  "verified_runtime_imports": {
    "numpy": "2.1.3",
    "pandas": "2.3.3",
    "scipy": "1.16.3",
    "sklearn": "1.6.1",
    "torch": "2.11.0+cu128",
    "tabpfn": "9.0.0"
  },
  "feature_count": 184,
  "text_feature_arm": "masked_sequence",
  "masked_feature_module_sha256": "ce6313975b0f8eb8a3b7c8ce9cd0d1656d0e16bd1f7340d50dbc42ec1ac991b2",
  "text_benchmark_source_sha256": "601f5124682320a970f4be51873d70b035f5c11fe3569ce244b0bad21a4dd665",
  "masked_svd_components_per_view": 16,
  "runtime_diagnostic_script_version_id": 359162368,
  "runtime_diagnostic_query_capacity_rows": 3,
  "larger_feature_capacity_reference": {
    "script_version_id": 359154729,
    "features": 216,
    "query_rows": 4096
  },
  "reference_scored_version": 2,
  "final_claude_gate": true,
  "baseline_balance_alpha": 0.5,
  "centered_stylegram_shrinkage": 1.0,
  "centered_stylegram_adapter_sha256": "4e1158f9d30d23f0cece8477f7330038edf9ee41e096aae78b5c6f006d955214",
  "parent_own_source_sha256": "b040fd59cb205a27e69b6a189a28c2c239e09a57b452ba1cccf7f2e88198c3a4",
  "reference_notebook_id": 139200100,
  "candidate_admitted": true,
  "protected_weight": 0.1,
  "protected_operator_sha256": "9d9f8a89b92935075131d1e3a26a7948cbbe6eb1495d5ca808329a5ecda3184a",
  "raw_extension": "smol_center_add"
}
'''[1:-1], crlf=True)


In [ ]:
'''Name That LLM | Upstream Module: optional-config.json
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('optional-config.json', r'''
{
  "candidate_admitted": true,
  "selected_by_root": true,
  "assets_public_audited": true,
  "diagnostic_PASS_binding": {
    "notebook_id": 139353455,
    "version": 3,
    "script_version_id": 359581988,
    "audit_sha256": "8e76708fb1cca9239ffabfe7e8cd029b954d3957577f034156c3cfda4a9ba525",
    "notebook_sha256": "8b95573e0fc61e61ee4ae96f443112ccff9a2c10df4929ec669d0e9b3c98b77d"
  },
  "raw_extension": "smol_center_add",
  "coefficient": 0.03,
  "alpha": 0.5,
  "protected_weight": 0.1,
  "post": 0,
  "compact_asset_ref": "lightningv08/ntl-smollm3-final-inference-assets/1",
  "compact_asset_id": 12671580,
  "compact_asset_manifest_sha256": "1772703faaee5d2b30e72fe37d0802c0c20eab2332e4d67a061cda6912fb611a",
  "base_ref": "lightningv08/smollm3-3b-base-offline-weights/1",
  "base_notebook_id": 139339821,
  "source_sha256": {
    "component.py": "ab1df94cea1358d95125ed99f4fbe9606f7e0690ca6b6938ead419f884343f46",
    "owned_path.py": "20dabafd3f423196863ce3d1fbd4caa8b196bd517a88422561615db003d1e620",
    "run_component.py": "8d6ee8631ffa99baaf3f9fdb49e3d8a4dbd6dbe2284e2ea1617ea5965902052b",
    "center_readout.py": "499b99f6ca94b6b4b8eb1fca15a4ef0519079194406441e70f8d0e2443068072",
    "source_balance.py": "da8682bc8298aed63c89607ceb08498a47a20ba7ab74752dddbdda996eab2351",
    "claude_gate.py": "3fd1971a10eb15df21dd2c25acf6516d22e7fb07708b4cc0d4a872681fb37ae4",
    "protected_operator.py": "9d9f8a89b92935075131d1e3a26a7948cbbe6eb1495d5ca808329a5ecda3184a",
    "run_parallel.py": "3ff30d27d811decbe0a19628a9933ae9a797f5191d303f7cd3257883ba66599f",
    "run_candidate_optional.py": "ab170cfa87abbfebcffb877a4b9db6b4cfa25944531b320e2e35da20822e013b",
    "compute_center.py": "3dbf7ca0c3769fbd4ade05346e4daeb791d589d7a7361e967f08d44c46999bbd",
    "finish_center_protected.py": "aa4e54e2fdc8ea4f209ae42138d6b808ac07c45bf232c68db0318c5df8da0a47",
    "admission_preflight.py": "59de3c41b2a1146a11797008b80e77859a0282b0d8b14c6a152fafc9dba19eeb"
  },
  "smol_timeout_seconds": 3600,
  "diagnostic_actual_PASS_bound": true,
  "root_decision_binding": {
    "explicit_candidate_admission": true,
    "selected_branch": "smol_center_add_protected_pfn10",
    "decision_sha256": "20e6dedbb78357c9b5714d7013a2d530de5ae8585651b04a724fb0170ee998cf",
    "joint_result_sha256": "f26285d0e463ed814de463cc190574d7d983923bb4087a38e423866e8ec73676",
    "public_assets_audit_sha256": "bbc3cb858630e5cb5d473c677d5c0744c8b688d41a4027fe647ec2c70f09f3dd"
  }
}
'''[1:-1], crlf=True)


In [ ]:
'''Name That LLM | Upstream Module: smol-spec.json
Preserved source dependency; verified against its original SHA-256.
'''
bundle.add('smol-spec.json', r'''
{
  "mode": "SOURCE_ONLY_OPTIONAL_SMOL_CENTER_ADD",
  "inference_authorized": true,
  "diagnostic_only_authorized": false,
  "actual_final_binding": {
    "actual_full_QA_passed": true,
    "paired_gain_admitted_by_root": true,
    "notebook_id": 139347470,
    "version": 1,
    "script_version_id": 359564738,
    "worker_sha256": "4dea00c2313dad426ba1081f3b571540c84f6e5e441c57075699e6f9557e935b",
    "checkpoint_relative": "final/finished_checkpoint",
    "complete_sha256": "31b52250f5eaab7cbd9b60152e4471455566b6d40fae911f0f1025ac827854fb",
    "report_sha256": "53be4d9394bd161ed0d58b75ecdc39f8dafb40e12dec8d9ad5b3e76d42caf784",
    "portable_files_sha256": {
      "adapter/adapter_model.safetensors": "acc77f2e95d86e0edc541afa46c1af16ae957bed955435c5319a4dc8a60b4c41",
      "adapter/adapter_config.json": "fe64dfa8471149025ec910d7a673c03ee673827041efc1cda0988dec16b42af3",
      "heads.safetensors": "cbc50ca2abc66a6002459a2343b02c0598e8d94fe0954303959d297ae2c2fe8b",
      "readouts.npz": "04115910b24569c5e08641342b0cefe93c8156515f93d0b920ede01dfba2ddc2",
      "replay_anchors.npz": "e49a747a32a5529f2d0e1363a3f5b7d9e1ca34d7fc114571ec624277207a29b2",
      "adapter/README.md": "436049a05f1f088661ddcdc33c42b1e139d5ed147de162b74b178c73db689bfb"
    },
    "full_qa_audit_sha256": "9742ec63fd13211f000c8e464c3e88e197eb032f8a085cb57b40e5d4fc154a7e"
  },
  "training_worker_sha256": "4dea00c2313dad426ba1081f3b571540c84f6e5e441c57075699e6f9557e935b",
  "expected_full_notebook_id": 139347470,
  "expected_full_notebook_version": 1,
  "base_revision": "d78a42f79198603e614095753484a04c10c2b940",
  "base_manifest_sha256": "120112bd002cde8bd7705514467081db4d78646c222b3958b6e3ab0d09098911",
  "base_files": {
    "README.md": {
      "size": 12340,
      "git_blob": "48b8618e0d20e47e826a0a842c78a2dc4616d6a9",
      "lfs_sha256": null
    },
    "config.json": {
      "size": 943,
      "git_blob": "975eb9a0acea3b739ecdbc4a4f615d688b27b407",
      "lfs_sha256": null
    },
    "generation_config.json": {
      "size": 126,
      "git_blob": "e0b85d8040e51ed5efddf6a2501c3a462b40e6da",
      "lfs_sha256": null
    },
    "model-00001-of-00002.safetensors": {
      "size": 4966315264,
      "git_blob": "81dcd69d7d543ca2f74efcb3750ee947231ed183",
      "lfs_sha256": "7e270ac568ee1880ddbadad66ccdcd9906d52415e8904e2f300c75250b9c7d49"
    },
    "model-00002-of-00002.safetensors": {
      "size": 1183919744,
      "git_blob": "0d3daab6805b3cd55fc1dd88f5f1d90bc66c183b",
      "lfs_sha256": "c6a6e7690a66dcc386a6a9b456686e7c308d45cba884d116c928dafa7fa987ae"
    },
    "model.safetensors.index.json": {
      "size": 26899,
      "git_blob": "ee6c028c012814d385be51c1d537c345c9fa9457",
      "lfs_sha256": null
    },
    "special_tokens_map.json": {
      "size": 151,
      "git_blob": "d2a1bbca9df5663dbd63c80560824f8aef02cf9e",
      "lfs_sha256": null
    },
    "tokenizer.json": {
      "size": 17208861,
      "git_blob": "0cd8b7fa7c1f17845e0d5e74f9d5afe90ca3105c",
      "lfs_sha256": "ab4da6b2aa68247e9c0fa9b97fc7fcc796505038d01f7e144522a65ce0dbd2e5"
    },
    "tokenizer_config.json": {
      "size": 50405,
      "git_blob": "c57d419f2a96470776f99fa37a9143363d3ae047",
      "lfs_sha256": null
    }
  },
  "core_versions": {
    "torch": "2.11.0+cu128",
    "transformers": "5.16.1",
    "peft": "0.20.0",
    "accelerate": "1.14.0",
    "safetensors": "0.8.0",
    "numpy": "2.1.3",
    "pandas": "2.3.3"
  },
  "classes": [
    "claude-fable-5",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    "glm-5.3",
    "glm-5.3-flash",
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    "kimi-k3",
    "qwen3.8-max"
  ],
  "train_sha256": "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707",
  "dtype": "FP16 SDPA base; FP32 adapters/norms/readouts; FP32 affine/log_softmax",
  "max_length": 448,
  "microbatch": 3,
  "pool_layers": [
    6,
    12,
    18,
    24,
    30,
    36
  ],
  "feature_width": 12288,
  "output_classes": 18,
  "first_token_excluded_from_mean": true,
  "test_id_order_preserved": true,
  "feature_replay_rtol": 1E-05,
  "feature_replay_atol": 1E-05,
  "probability_replay_maxabs": 1E-06,
  "training": false,
  "optimizer_state_loaded": false,
  "targets_loaded": false,
  "submission_authorized": false,
  "source_hashes": {
    "component.py": "ab1df94cea1358d95125ed99f4fbe9606f7e0690ca6b6938ead419f884343f46",
    "owned_path.py": "20dabafd3f423196863ce3d1fbd4caa8b196bd517a88422561615db003d1e620",
    "run_component.py": "8d6ee8631ffa99baaf3f9fdb49e3d8a4dbd6dbe2284e2ea1617ea5965902052b"
  }
}
'''[1:-1], crlf=True)


In [ ]:
'''Name That LLM | Experiment Imports
'''

import hashlib
import json
from pathlib import Path
import sys

import numpy as np


In [ ]:
'''Name That LLM | ConfidenceCentering
'''
class ConfidenceCentering:
    def __init__(self, strength=0.25):
        self.strength = strength
        self.mixed_weights = None
        self.style_weights = None
        self.audit = {}

    def weights_from_distances(self, distances, groups, threshold):
        if threshold <= 0 or not 0 <= self.strength <= 1:
            raise ValueError('Invalid distance threshold or attenuation strength')
        groups = np.asarray(groups)
        distances = np.asarray(distances)
        if distances.shape != (len(groups), len(groups)) or not np.isfinite(distances).all():
            raise ValueError('Expected a finite square pairwise distance matrix')
        weights = np.ones(len(groups), dtype=np.float64)
        for group in np.unique(groups):
            rows = np.flatnonzero(groups == group)
            if not 2 <= len(rows) <= 18:
                continue
            block = distances[np.ix_(rows, rows)]
            pairs = block[np.triu_indices(len(rows), 1)]
            weights[rows] = 1 - self.strength * np.mean(pairs > threshold)
        return weights

    def observe_stylegram(self, distances, groups, threshold):
        self.style_weights = self.weights_from_distances(distances, groups, threshold)
        self.audit['stylegram'] = self._describe(self.style_weights)

    def observe_mixed(self, texts, embeddings, groups, adaptive, threshold):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.mixed_weights = np.ones(len(texts), dtype=np.float64)
        if not np.asarray(adaptive).any():
            self.audit['mixed'] = self._describe(self.mixed_weights)
            return
        if embeddings is None:
            raise ValueError('The confidence variant requires the reference GLM clustering features')
        text_features = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit_transform(texts)
        semantic = np.asarray(embeddings, dtype=np.float64)
        semantic = semantic - semantic.mean(0)
        semantic /= np.linalg.norm(semantic, axis=1, keepdims=True) + 1e-12
        for group in np.unique(groups[np.asarray(adaptive)]):
            rows = np.flatnonzero(groups == group)
            text_distance = np.clip(1 - (text_features[rows] @ text_features[rows].T).toarray(), 0, None)
            semantic_distance = np.clip(1 - semantic[rows] @ semantic[rows].T, 0, None)
            mixed = (text_distance + semantic_distance) / 2
            local = self.weights_from_distances(mixed, np.zeros(len(rows), dtype=int), threshold)
            self.mixed_weights[rows] = local
        self.audit['mixed'] = self._describe(self.mixed_weights)

    @staticmethod
    def _describe(weights):
        return {'rows': len(weights), 'attenuated_rows': int((weights < 1).sum()),
                'min_weight': float(weights.min()) if len(weights) else 1.0,
                'mean_weight': float(weights.mean()) if len(weights) else 1.0}


In [ ]:
'''Name That LLM | ProtectedFamilyBlend
'''
class ProtectedFamilyBlend:
    @staticmethod
    def apply(base, predictions, conditional, indices, alpha=0.25):
        base = np.asarray(base, dtype=float)
        predictions = np.asarray(predictions)
        conditional = np.asarray(conditional, dtype=float)
        indices = np.asarray(indices, dtype=int)
        if (base.ndim != 2 or conditional.shape != (len(base), len(indices))
                or predictions.shape != (len(base),) or not 0 <= alpha <= 1):
            raise ValueError('Invalid family blend inputs')
        if (not np.isfinite(base).all() or not np.isfinite(conditional).all()
                or (base < 0).any() or (conditional < 0).any()
                or not np.allclose(base.sum(1), 1, atol=1e-6)
                or not np.allclose(conditional.sum(1), 1, atol=1e-6)):
            raise ValueError('Family blending requires valid probability rows')
        output = base.copy()
        final_predictions = predictions.copy()
        rows = np.flatnonzero(np.isin(predictions, indices))
        mass = base[np.ix_(rows, indices)].sum(1, keepdims=True)
        output[np.ix_(rows, indices)] = (
            (1 - alpha) * base[np.ix_(rows, indices)] + alpha * mass * conditional[rows]
        )
        candidate = output.argmax(1)
        crossed = rows[~np.isin(candidate[rows], indices)]
        output[crossed] = base[crossed]
        eligible = rows[np.isin(candidate[rows], indices)]
        final_predictions[eligible] = candidate[eligible]
        return output, final_predictions, len(crossed)


In [ ]:
'''Name That LLM | FrozenFamilySpecialist
'''
class FrozenFamilySpecialist:
    '''shortcut: validate on a frozen-Qwen proxy; upgrade when aligned full-ensemble OOF is available.'''

    def __init__(self, alpha=0.25, output_dir=Path('/kaggle/working')):
        self.alpha = alpha
        self.output_dir = Path(output_dir)
        self.selected = None
        self.conditional = None
        self.indices = None
        self.labels = None
        self.audit = {'validation': 'frozen Qwen3-8B head proxy; not full ensemble OOF',
                      'selection_folds': [1, 2, 3, 4], 'confirmation_fold': 0,
                      'alpha': alpha, 'admitted': False}

    def wrap_fit_head(self, original, train, labels):
        def fitted(key, train_features, test_features, regularization):
            result = original(key, train_features, test_features, regularization)
            if key == 'q3_8b_cat7+22':
                self.fit(train_features, test_features, train, labels, regularization)
            return result
        return fitted

    @staticmethod
    def _align(model, features, indices):
        order = [list(model.classes_).index(index) for index in indices]
        return model.predict_proba(features)[:, order]

    @staticmethod
    def _query_scale(features, scaler):
        return ((features - features.mean(0)) / (features.std(0) + 1e-6)
                if len(features) >= 2 else scaler.transform(features))

    def fit(self, train_features, test_features, train, labels, regularization):
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import balanced_accuracy_score, confusion_matrix
        from sklearn.model_selection import GroupKFold
        from sklearn.preprocessing import StandardScaler
        from threadpoolctl import threadpool_limits
        self.labels = list(labels)
        targets = train['model'].map({label: i for i, label in enumerate(labels)}).to_numpy()
        groups = train['prompt_id'].to_numpy()
        if len(np.unique(groups)) < 5:
            raise ValueError('Five disjoint prompt folds are required for specialist selection')
        families = {name: np.array([i for i, label in enumerate(labels) if label.startswith(prefix)])
                    for name, prefix in [('openai', 'gpt-'), ('gemini', 'gemini-')]}
        self.audit['folds'] = []
        oof = np.zeros((len(train), len(labels)), dtype=float)
        oof_family = {name: np.zeros((len(train), len(idx)), dtype=float) for name, idx in families.items()}
        fold_ids = np.full(len(train), -1, dtype=int)
        with threadpool_limits(limits=4):
            for fold, (fit_rows, valid_rows) in enumerate(GroupKFold(5).split(train_features, targets, groups)):
                if set(groups[fit_rows]) & set(groups[valid_rows]):
                    raise ValueError('Prompt leakage in specialist validation')
                scaler = StandardScaler().fit(train_features[fit_rows])
                fit_features = scaler.transform(train_features[fit_rows])
                query_features = self._query_scale(train_features[valid_rows], scaler)
                general = LogisticRegression(C=regularization, max_iter=5000).fit(fit_features, targets[fit_rows])
                proxy = self._align(general, query_features, np.arange(len(labels)))
                base_predictions = proxy.argmax(1)
                base_score = balanced_accuracy_score(targets[valid_rows], base_predictions)
                oof[valid_rows] = proxy
                fold_ids[valid_rows] = fold
                record = {'fold': fold, 'valid_rows': len(valid_rows), 'baseline': base_score, 'families': {}}
                for name, indices in families.items():
                    inside = np.isin(targets[fit_rows], indices)
                    specialist = LogisticRegression(C=regularization, max_iter=5000).fit(
                        fit_features[inside], targets[fit_rows][inside])
                    conditional = self._align(specialist, query_features, indices)
                    oof_family[name][valid_rows] = conditional
                    _, candidate, _ = ProtectedFamilyBlend.apply(proxy, base_predictions, conditional, indices, self.alpha)
                    score = balanced_accuracy_score(targets[valid_rows], candidate)
                    record['families'][name] = {'score': score, 'delta': score - base_score}
                self.audit['folds'].append(record)
                print('Specialist proxy fold:', json.dumps(record), flush=True)
            selection = {name: float(np.mean([r['families'][name]['delta']
                                             for r in self.audit['folds'] if r['fold'] != 0]))
                         for name in families}
            chosen = max(selection, key=selection.get)
            confirmation = self.audit['folds'][0]['families'][chosen]['delta']
            self.audit.update(selection_mean_deltas=selection, selected_on_development=chosen,
                              confirmation_delta=confirmation)
            if selection[chosen] > 0 and confirmation > 0:
                self.selected = chosen
                self.indices = families[chosen]
                scaler = StandardScaler().fit(train_features)
                inside = np.isin(targets, self.indices)
                specialist = LogisticRegression(C=regularization, max_iter=5000).fit(
                    scaler.transform(train_features)[inside], targets[inside])
                self.conditional = self._align(specialist, self._query_scale(test_features, scaler), self.indices)
                self.audit['admitted'] = True
        self.audit['proxy_confusion_matrix'] = confusion_matrix(targets, oof.argmax(1), labels=np.arange(len(labels))).tolist()
        self.audit['labels'] = list(labels)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(self.output_dir / 'family_proxy_oof.npz',
                            ids=train['id'].to_numpy(dtype=str), prompt_ids=groups.astype(str),
                            labels=np.asarray(labels, dtype=str), targets=targets, folds=fold_ids,
                            general=oof, **oof_family)

    def refine(self, probabilities, predictions):
        if self.labels is None:
            raise RuntimeError('The frozen Qwen feature hook was not executed')
        result = predictions.copy()
        if self.selected is not None:
            _, result, reverted = ProtectedFamilyBlend.apply(
                probabilities, predictions, self.conditional, self.indices, self.alpha)
            self.audit['reverted_family_crossings'] = reverted
        self.audit['changed_predictions'] = int((result != predictions).sum())
        self.audit['selected_family'] = self.selected
        (self.output_dir / 'family_specialist_audit.json').write_text(json.dumps(self.audit, indent=2))
        if self.selected is None:
            print('No specialist passed both development and confirmation. Output equals the reference; do not spend a submission on a duplicate.', flush=True)
        else:
            print('Admitted specialist:', self.selected, 'changed predictions:', self.audit['changed_predictions'], flush=True)
        return result


In [ ]:
'''Name That LLM | ReferenceIntervention
'''
class ReferenceIntervention:
    def __init__(self, mode):
        if mode not in ('cluster', 'family'):
            raise ValueError(mode)
        self.mode = mode
        self.centering = ConfidenceCentering()
        self.specialist = FrozenFamilySpecialist()
        self.patches = []

    def replace(self, source, old, new, name):
        if source.count(old) != 1:
            raise ValueError(f'Expected exactly one source anchor for {name}')
        self.patches.append(name)
        return source.replace(old, new, 1)

    def transform(self, source):
        if self.mode == 'cluster':
            old = "    return fcluster(linkage(squareform(_dt, checks=False), method='average'),\n                    float(threshold), criterion='distance')"
            new = "    _groups = fcluster(linkage(squareform(_dt, checks=False), method='average'),\n                       float(threshold), criterion='distance')\n    _intervention.centering.observe_stylegram(_dt, _groups, threshold)\n    return _groups"
            source = self.replace(source, old, new, 'capture_stylegram_cluster_distances')
            source = self.replace(source, '        p[_stylegram_rows] = _p_center',
                '        _q = _intervention.centering.style_weights[_stylegram_rows, None]\n        p[_stylegram_rows] = _q * _p_center + (1 - _q) * p[_stylegram_rows]', 'attenuate_stylegram_replacement')
            old = '    ADAPT_ROWS = ADAPT & (sz >= ADAPT_MIN) & (sz <= ADAPT_MAX)'
            source = self.replace(source, old, old + '\n    _intervention.centering.observe_mixed(TE, E_te, G_TE, ADAPT_ROWS, CLUSTER_THRESH_MIX)', 'capture_mixed_cluster_confidence')
            old = 'LOGIT[ADAPT_ROWS] += TWIN_W[key] * np.log(np.clip(pc, 1e-9, 1))'
            source = self.replace(source, old, 'LOGIT[ADAPT_ROWS] += (TWIN_W[key] * np.log(np.clip(pc, 1e-9, 1))) * _intervention.centering.mixed_weights[ADAPT_ROWS, None]', 'attenuate_frozen_centered_heads')
            old = 'LOGIT[ADAPT_ROWS] += TWIN_W_LORA[key] * (lc - np.log(np.exp(lc).sum(1, keepdims=True)))'
            source = self.replace(source, old, 'LOGIT[ADAPT_ROWS] += (TWIN_W_LORA[key] * (lc - np.log(np.exp(lc).sum(1, keepdims=True)))) * _intervention.centering.mixed_weights[ADAPT_ROWS, None]', 'attenuate_lora_centered_heads')
        else:
            old = 'for key, mt, hid, layers, C, smoke_id, pool in LLM_HEADS:'
            source = self.replace(source, old, 'fit_head = _intervention.specialist.wrap_fit_head(fit_head, train, LABELS)\n' + old, 'capture_frozen_qwen_features')
            old = "sub = pd.DataFrame({'id': test['id'], 'model': [LABELS[i] for i in pred]})"
            source = self.replace(source, old,
                "pd.DataFrame({'id': test['id'], 'model': [LABELS[i] for i in pred]}).to_parquet('family_reference_control.parquet', index=False)\npred = _intervention.specialist.refine(P, pred)\n" + old,
                'apply_confirmed_family_specialist')
        compile(source, 'intervened_reference', 'exec')
        return source

    def run(self, directory=None):
        here = Path(directory) if directory is not None else Path(__file__).resolve().parent
        original = (here / 'ntl_reference.py').read_bytes()
        expected = json.loads((here / 'runtime_config.json').read_text())['reference_source_sha256']
        if hashlib.sha256(original).hexdigest() != expected:
            raise ValueError('The reference source has changed')
        source = self.transform(original.decode())
        namespace = {'__name__': '__main__', '__file__': str(here / 'ntl_reference.py'), '_intervention': self}
        exec(compile(source, 'intervened_reference', 'exec'), namespace)
        audit = {'variant': self.mode, 'reference_sha256': expected,
                 'executed_sha256': hashlib.sha256(source.encode()).hexdigest(), 'patches': self.patches,
                 'full_ensemble_oof_verified': False,
                 'centering': self.centering.audit if self.mode == 'cluster' else None}
        Path('/kaggle/working/intervention_audit.json').write_text(json.dumps(audit, indent=2))


In [ ]:
'''Name That LLM | OOP Experiment Runner
'''
class ImprovementExperiment(ReferenceEnsembleExperiment):
    def __init__(self, config, bundle, mode):
        super().__init__(config, bundle)
        self.mode = mode

    def run(self):
        self.bundle.materialize(self.config.source_dir)
        data, test, labels = KagglePreflight(self.config, self.bundle).check()
        self.config.work_dir.mkdir(parents=True, exist_ok=True)
        if self.artifact.path.exists():
            raise FileExistsError('submission.parquet already exists; start a clean session')
        old_directory = Path.cwd()
        started = time.monotonic()
        intervention = ReferenceIntervention(self.mode)
        os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')
        try:
            os.chdir(self.config.work_dir)
            intervention.run(self.config.source_dir)
            submission = self.artifact.validate(test, labels)
        except BaseException:
            self.artifact.quarantine()
            raise
        finally:
            os.chdir(old_directory)
        summary = {
            'experiment': self.config.name, 'variant': self.mode,
            'base_notebook': '03_public_reference_oop.ipynb',
            'base_score_mapping_confirmed': False,
            'reference_sha256': self.bundle.digests['ntl_reference.py'],
            'experiment_patches': intervention.patches,
            'source_bytes_preserved': True, 'numerical_execution_modified': True,
            'test_rows': len(test), 'elapsed_seconds': time.monotonic() - started,
            'submission_sha256': hashlib.sha256(self.artifact.path.read_bytes()).hexdigest(),
            'score_verified': False, 'full_ensemble_oof_verified': False,
        }
        (self.config.work_dir / 'reproduction_summary.json').write_text(json.dumps(summary, indent=2))
        print(json.dumps(summary, indent=2))
        return submission


In [ ]:
'''Name That LLM | Configure the Independent Experiment
'''
config = ExperimentConfig(name='ntl_05_cluster', full_ensemble=False)
experiment = ImprovementExperiment(config, bundle, mode='cluster')
bundle.verify()
print(f"Frozen reference integrity verified: {len(bundle.sources)} files")


In [ ]:
'''Name That LLM | Run and Validate Submission
This launches the actual neural reference with one audited intervention.
Inspect intervention_audit.json and, for the family experiment,
family_specialist_audit.json before spending a leaderboard attempt.
'''
submission = experiment.run()
submission.head()
